<style>
  @import url('https://fonts.googleapis.com/css2?family=Montserrat:wght@400;600;700&display=swap');
</style>

<div style="font-family:'Montserrat', ui-sans-serif, system-ui, -apple-system, 'Segoe UI', Roboto, Helvetica, Arial; padding:10px 0 6px 0; border-bottom:1px solid #ef4444;">
  <div style="display:flex; align-items:flex-start; gap:12px;">
    <img src="../assets/aiphet-logo.png" alt="Aiphet logo" style="width:44px;height:44px;border-radius:10px;object-fit:contain; margin-top:2px;" />
    <div style="line-height:1.15;">
      <div style="font-size:26px;font-weight:700;letter-spacing:0.2px;">Aiphet</div>
      <div style="font-size:14px;color:#4b5563;margin-top:2px;">Fine-tuning (LoRA/QLoRA) of a public Qwen model for high-quality PROMs, PREMs, and general medical forms</div>
      <div style="font-size:12px;color:#6b7280;margin-top:6px;">Author: Pablo Pimàs</div>
      <div style="font-size:12px;color:#6b7280;margin-top:2px;">Email: pablo@pimas.cat</div>
      <div style="font-size:12px;color:#6b7280;margin-top:2px;">Date: February 22, 2026</div>
      <div style="font-size:12px;color:#6b7280;margin-top:2px;">License: CC BY 4.0 (SPDX: CC-BY-4.0)</div>
      <div style="font-size:12px;color:#6b7280; font-weight:600; margin-top:2px;">#AIP-173</div>
    </div>
  </div>
</div>


## Table of Contents

1. Fine-Tuning a Base Instruct Model for FHIR R4 Questionnaire Generation
2. Reproducibility Protocol
3. Model Selection and Multi-Model Reuse
4. Dataset Statement and Governance
5. Data Loading and Prompt/Completion Parsing
6. FHIR Structural Audit
7. Training Configuration and Rationale
8. Reproducible Train/Validation Split
9. Data Materialization for Training
10. Training Execution (MLX-LM)
11. Energy Accounting and Carbon Estimation
12. GGUF Deliverable for LM Studio
13. Adapter Inference Test
14. Baseline Inference Test
15. A/B Check on Validation Samples
16. Rule-based A/B Evaluation
17. Post-analysis: Strict vs Relaxed Criteria
18. Adapter Validation Protocol (Base vs Adapter)
19. Go/No-Go Decision from Saved Artifacts
20. Quantitative Evaluation
21. Qualitative Analysis
22. Limitations, Ethics, and Threats to Validity
23. References
24. How to Cite

# Fine-Tuning MLX models for FHIR R4 Questionnaire Generation

## Scope

This notebook documents an end-to-end, reproducible training workflow for adapting **the model selected via `AIPROM_MODEL_NAME`** to generate **complete FHIR R4 Questionnaire JSON** from instruction prompts in a clinical PROMs/PREMs context.

The workflow is designed for:

- Apple Silicon execution using MLX/MLX-LM
- parameter-efficient adaptation (LoRA/QLoRA-style setup)
- structured-output generation under FHIR-oriented constraints

## Main Contributions

1. Reproducible data preparation and validation pipeline for synthetic prompt/completion JSONL records.
2. Structured fine-tuning protocol for complete FHIR Questionnaire generation.
3. Quantitative evaluation of structural validity and schema-conformance behavior.
4. Qualitative error analysis of generated medical questionnaire forms.
5. Explicit legal/licensing caveats for standardized instrument content.

## Research Context

Patient-reported outcomes (PROMs) and patient-reported experience measures (PREMs) often require strict structural interoperability.  
This notebook focuses on generating machine-usable complete forms compatible with **HL7 FHIR R4 Questionnaire semantics**.

## References (Core Concepts)

- HL7 FHIR R4 Questionnaire: https://hl7.org/fhir/R4/questionnaire.html
- LoRA (Hu et al., 2021): https://arxiv.org/abs/2106.09685
- QLoRA (Dettmers et al., 2023): https://arxiv.org/abs/2305.14314
- Qwen documentation: https://qwen.readthedocs.io/
- MLX: https://github.com/ml-explore/mlx
- MLX-LM: https://github.com/ml-explore/mlx-lm

## Compliance Note

This notebook is provided for research and educational use.  
If standardized questionnaire text is redistributed or used commercially, licensing and copyright obligations must be verified instrument-by-instrument.

## Reproducibility Protocol

To support repeatability, this notebook follows a deterministic workflow where possible.

### Execution Environment

- Platform target: Apple Silicon (macOS) with MLX/MLX-LM
- Python environment managed with pinned package versions from `requirements.txt`
- Notebook and scripts executed from repository root to preserve relative paths

### Determinism Settings

We will enforce and report:

- global random seed
- dataset split seed
- explicit train/validation partition method
- fixed configuration values for model and optimization

> Note: exact bitwise reproducibility is not always guaranteed across hardware/software backends, but seed and environment control substantially reduce variance.

### Experimental Tracking

For each training run, we will record:

1. model identifier
2. dataset file path and row counts
3. preprocessing/validation checks
4. hyperparameters (learning rate, LoRA rank, batch size, sequence length, steps)
5. runtime metadata (device, library versions)
6. evaluation outputs and qualitative examples

### Run Order

This notebook should be executed top-to-bottom in a single pass:

1. environment checks
2. dataset loading and validation
3. split generation
4. training
5. evaluation and error analysis

### Energy Consumption

To improve sustainability reporting, this workflow tracks energy use and estimated emissions for key steps (especially training and inference) using `codecarbon` in offline mode.

Tracked artifacts are stored in `lab/artifacts/energy/` and include:

- step-level JSON summaries (duration, kWh, kgCO2eq, return code),
- consolidated CSV logs from CodeCarbon for later analysis and reporting.

### References

- ACM Artifact Review and Badging: https://www.acm.org/publications/policies/artifact-review-and-badging-current
- The Turing Way (Reproducibility): https://the-turing-way.netlify.app/reproducible-research/reproducible-research
- MLX: https://github.com/ml-explore/mlx

In [1]:
# Global imports + .env loading
from __future__ import annotations

import importlib
import json
import os
import platform
import random
import re
import shutil
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

cwd = Path.cwd().resolve()
env_path = next((p / ".env" for p in [cwd, *cwd.parents] if (p / ".env").exists()), None)

if env_path:
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ[key.strip()] = value.strip().strip('"').strip("'")

print(f".env: {env_path if env_path else 'not found'}")
print("AIPROM_MODEL_NAME:", os.getenv("AIPROM_MODEL_NAME", "<missing>"))
print("AIPROM_MODEL_BACKEND:", os.getenv("AIPROM_MODEL_BACKEND", "<missing>"))

.env: /Users/CAE9/aiprom-llm/.env
AIPROM_MODEL_NAME: mlx-community/Qwen2.5-3B-Instruct-4bit
AIPROM_MODEL_BACKEND: mlx_lm


In [2]:
# Reproducibility: environment snapshot + model selection

# --- Fixed seeds for reproducibility ---
GLOBAL_SEED = 173
random.seed(GLOBAL_SEED)
os.environ["PYTHONHASHSEED"] = str(GLOBAL_SEED)

# --- Helper to get package versions safely ---
def safe_version(module_name: str) -> str:
    """Return the installed version for a module, or a fallback label.

    Parameters
    ----------
    module_name:
        Importable module name (e.g., "mlx", "datasets").

    Returns
    -------
    str
        Module version string when available, otherwise "unknown" or
        "not-installed" if import fails.
    """
    try:
        module = importlib.import_module(module_name)
        return getattr(module, "__version__", "unknown")
    except Exception:
        return "not-installed"


# --- Detect repository root heuristically ---
def find_repo_root(start: Path) -> Path:
    """Find repository root by locating folders that match this project layout.

    The function walks from the current working directory up to parent folders
    and returns the first directory containing both `README.md` and `data/`.

    Parameters
    ----------
    start:
        Initial path used as search origin.

    Returns
    -------
    Path
        Detected repository root; if none is found, returns `start.resolve()`.
    """
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    return start.resolve()


def slugify_model_name(model_name: str) -> str:
    """Convert model identifier into a filesystem-safe artifact suffix."""
    slug = re.sub(r"[^a-zA-Z0-9._-]+", "-", model_name.strip())
    return slug.strip("-._").lower() or "default-model"


repo_root = find_repo_root(Path.cwd())
dataset_path = repo_root / "data" / "synthetic-aiprom-1500-firh4.jsonl"

# --- Model/backend selection (single source of truth for the full notebook) ---
MODEL_BACKEND = os.getenv("AIPROM_MODEL_BACKEND", "mlx_lm").strip().lower()
MODEL_NAME = os.getenv("AIPROM_MODEL_NAME", "org/model-instruct").strip()
MODEL_ALIAS = os.getenv("AIPROM_MODEL_ALIAS", slugify_model_name(MODEL_NAME))

if MODEL_BACKEND != "mlx_lm":
    raise ValueError(
        f"Unsupported MODEL_BACKEND={MODEL_BACKEND!r} for this notebook. "
        "Current workflow supports only 'mlx_lm'."
    )

artifacts_dir = repo_root / "lab" / "artifacts" / MODEL_ALIAS
artifacts_dir.mkdir(parents=True, exist_ok=True)

# --- System/runtime info ---
runtime_info = {
    "timestamp_utc": datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "cwd": str(Path.cwd().resolve()),
    "repo_root": str(repo_root),
    "dataset_exists": dataset_path.exists(),
    "dataset_path": str(dataset_path),
    "seed": GLOBAL_SEED,
    "model_backend": MODEL_BACKEND,
    "model_name": MODEL_NAME,
    "model_alias": MODEL_ALIAS,
    "artifacts_dir": str(artifacts_dir),
    "packages": {
        "mlx": safe_version("mlx"),
        "mlx_lm": safe_version("mlx_lm"),
        "datasets": safe_version("datasets"),
        "yaml": safe_version("yaml"),
        "numpy": safe_version("numpy"),
        "pandas": safe_version("pandas"),
        "wandb": safe_version("wandb"),
    },
}

# --- Optional: git metadata if available ---
if shutil.which("git"):
    try:
        commit = subprocess.check_output(
            ["git", "-C", str(repo_root), "rev-parse", "HEAD"],
            text=True,
        ).strip()
    except Exception:
        commit = "unavailable"
else:
    commit = "git-not-found"

runtime_info["git_commit"] = commit

print(json.dumps(runtime_info, indent=2))

/Users/CAE9/aiprom-llm/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "timestamp_utc": "2026-02-24T19:07:52Z",
  "python_version": "3.14.3",
  "platform": "macOS-26.3-arm64-arm-64bit-Mach-O",
  "machine": "arm64",
  "processor": "arm",
  "cwd": "/Users/CAE9/aiprom-llm/lab",
  "repo_root": "/Users/CAE9/aiprom-llm",
  "dataset_exists": true,
  "dataset_path": "/Users/CAE9/aiprom-llm/data/synthetic-aiprom-1500-firh4.jsonl",
  "seed": 173,
  "model_backend": "mlx_lm",
  "model_name": "mlx-community/Qwen2.5-3B-Instruct-4bit",
  "model_alias": "mlx-community-qwen2.5-3b-instruct-4bit",
  "artifacts_dir": "/Users/CAE9/aiprom-llm/lab/artifacts/mlx-community-qwen2.5-3b-instruct-4bit",
  "packages": {
    "mlx": "unknown",
    "mlx_lm": "0.30.7",
    "datasets": "4.5.0",
    "yaml": "6.0.3",
    "numpy": "2.4.2",
    "pandas": "3.0.1",
    "wandb": "0.25.0"
  },
  "git_commit": "70058a4d5336c8f35aa3b351b0c752cf16f0ed41"
}


## Dataset Statement and Governance

### Primary Dataset

This study uses the synthetic prompt-completion JSONL dataset:

- `data/synthetic-aiprom-1500-firh4.jsonl`

Each row contains:

- `prompt`: natural-language instruction for form generation
- `completion`: JSON string representing a **complete FHIR R4 Questionnaire**

### Structural Coverage

The target payloads include full questionnaire-level fields (`resourceType`, `id`, `status`, `title`, `item`) and nested item trees with mixed FHIR item types.

### Data Format Contract

The training objective is conditioned generation:

- **Input**: instruction prompt transformed into ChatML user turn
- **Output**: assistant JSON for a full FHIR `Questionnaire`

### Licensing and Usage Caveat

This notebook is for research/educational purposes.  
Some questionnaire content may correspond to third-party instruments with distinct copyright or licensing terms.

Before redistribution or commercial use, verify rights instrument-by-instrument.

See repository documentation for details:

- `README.md` (licensing overview and practical implications)
- `docs/datasets.md` (dataset scope and legal notes)

### References

- FAIR principles for scientific data: https://www.go-fair.org/fair-principles/
- PROMIS (HealthMeasures): https://www.healthmeasures.net/explore-measurement-systems/promis

## Model Selection and Multi-Model Reuse

This notebook is parameterized to run end-to-end from a single model selection block at the beginning.

### Selection variables (first code cell)

- `MODEL_BACKEND` (currently supported: `mlx_lm`),
- `MODEL_NAME` (e.g., `org/model-instruct`),
- `MODEL_ALIAS` (filesystem-safe name used for artifact isolation).

### Artifact isolation by model

All generated files are written under:

- `lab/artifacts/<MODEL_ALIAS>/...`

This avoids collisions between checkpoints, logs, A/B reports, and exports when comparing multiple models.

### Re-run guidance

When switching model:

1. Set `AIPROM_MODEL_NAME` (and optionally `AIPROM_MODEL_ALIAS`) in environment,
2. run the first reproducibility/config cell,
3. re-run the workflow sections as needed (training or only evaluation).

This design allows reproducible side-by-side experiments with different compatible MLX models.

In [3]:
# Data loading: prompt/completion JSONL -> ChatML text + parsed Questionnaire JSON

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any


SYSTEM_PROMPT = (
    "You are a FHIR R4 expert. Return ONLY raw JSON (no markdown, no code fences). "
    "Generate a complete FHIR Questionnaire object."
)


@dataclass
class RecordParseResult:
    row_id: int
    raw_record: dict[str, Any]
    prompt: str
    completion_raw: str
    text: str
    assistant_obj: dict[str, Any]


def completion_to_json_string(completion: Any) -> str:
    """Normalize completion to a JSON string."""
    if isinstance(completion, str):
        value = completion.strip()
        if not value:
            raise ValueError("Empty completion string")
        return value
    if isinstance(completion, dict):
        return json.dumps(completion, ensure_ascii=False)
    raise ValueError("Completion must be a JSON string or object")


def build_chatml_text(prompt: str, completion_json: str) -> str:
    """Build deterministic ChatML transcript expected by MLX-LM text datasets."""
    prompt_clean = prompt.strip()
    completion_clean = completion_json.strip()
    if not prompt_clean:
        raise ValueError("Prompt is empty")
    if not completion_clean:
        raise ValueError("Completion JSON is empty")

    return (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}\n"
        "<|im_end|>"
        "<|im_start|>user\n"
        f"{prompt_clean}\n"
        "<|im_end|>"
        "<|im_start|>assistant\n"
        f"{completion_clean}\n"
        "<|im_end|>"
    )


def load_fhir_jsonl_records(path: Path) -> tuple[list[RecordParseResult], list[str]]:
    """Load and parse JSONL rows for full Questionnaire training."""
    results: list[RecordParseResult] = []
    errors: list[str] = []

    with path.open("r", encoding="utf-8") as f:
        for row_id, line in enumerate(f, start=1):
            if not line.strip():
                errors.append(f"Line {row_id}: empty line")
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                errors.append(f"Line {row_id}: invalid JSONL record ({exc})")
                continue

            if not isinstance(record, dict):
                errors.append(f"Line {row_id}: record is not a JSON object")
                continue

            prompt = record.get("prompt")
            if not isinstance(prompt, str) or not prompt.strip():
                errors.append(f"Line {row_id}: missing or invalid 'prompt' field")
                continue

            try:
                completion_raw = completion_to_json_string(record.get("completion"))
            except ValueError as exc:
                errors.append(f"Line {row_id}: {exc}")
                continue

            try:
                assistant_obj = json.loads(completion_raw)
            except json.JSONDecodeError as exc:
                errors.append(f"Line {row_id}: completion JSON decode error ({exc})")
                continue

            if not isinstance(assistant_obj, dict):
                errors.append(f"Line {row_id}: completion payload is not a JSON object")
                continue

            try:
                text = build_chatml_text(prompt, completion_raw)
            except ValueError as exc:
                errors.append(f"Line {row_id}: {exc}")
                continue

            results.append(
                RecordParseResult(
                    row_id=row_id,
                    raw_record=record,
                    prompt=prompt,
                    completion_raw=completion_raw,
                    text=text,
                    assistant_obj=assistant_obj,
                )
            )

    return results, errors


dataset_path = Path(runtime_info["dataset_path"])
parsed_records, parse_errors = load_fhir_jsonl_records(dataset_path)

print(f"Dataset path: {dataset_path}")
print(f"Parsed records: {len(parsed_records)}")
print(f"Parse errors: {len(parse_errors)}")

if parse_errors:
    print("\nSample parse errors (up to 10):")
    for err in parse_errors[:10]:
        print(f"- {err}")

if parsed_records:
    sample = parsed_records[0].assistant_obj
    print("\nFirst completion object keys:", sorted(sample.keys()))
    print("Sample prompt:", parsed_records[0].prompt[:140])

Dataset path: /Users/CAE9/aiprom-llm/data/synthetic-aiprom-1500-firh4.jsonl
Parsed records: 1500
Parse errors: 0

First completion object keys: ['code', 'date', 'id', 'item', 'resourceType', 'status', 'subjectType', 'title']
Sample prompt: Generate complete FHIR R4 Questionnaire for PHQ-9 Depression Assessment


In [4]:
# FHIR structural audit: complete Questionnaire checks and coverage summary

from __future__ import annotations

from collections import Counter
from dataclasses import dataclass
from typing import Any


FHIR_ALLOWED_TYPES = {
    "group",
    "display",
    "boolean",
    "decimal",
    "integer",
    "date",
    "dateTime",
    "time",
    "string",
    "text",
    "url",
    "choice",
    "open-choice",
}


@dataclass
class AuditSummary:
    total_records: int
    valid_records: int
    invalid_records: int
    pass_rate: float
    type_counts: dict[str, int]
    issue_counts: dict[str, int]
    avg_items_per_form: float


def is_valid_code_array(value: Any) -> bool:
    """Check whether `code` follows a minimal valid Coding[] shape."""
    if not isinstance(value, list) or len(value) == 0:
        return False

    for coding in value:
        if not isinstance(coding, dict):
            return False
        system = coding.get("system")
        code_value = coding.get("code")
        if not isinstance(system, str) or not system.strip():
            return False
        if not isinstance(code_value, str) or not code_value.strip():
            return False

    return True


def has_valid_answer_option(item: dict[str, Any]) -> bool:
    """Validate answerOption for choice/open-choice items."""
    answer_option = item.get("answerOption")
    if not isinstance(answer_option, list) or len(answer_option) == 0:
        return False

    for option in answer_option:
        if not isinstance(option, dict):
            return False
        if not any(str(key).startswith("value") for key in option.keys()):
            return False

    return True


def iter_questionnaire_items(items: Any) -> list[dict[str, Any]]:
    """Flatten questionnaire items recursively."""
    flat: list[dict[str, Any]] = []

    def _walk(nodes: Any) -> None:
        """Recursively traverse nested item arrays and collect item nodes."""
        if not isinstance(nodes, list):
            return
        for node in nodes:
            if not isinstance(node, dict):
                continue
            flat.append(node)
            child = node.get("item")
            if isinstance(child, list) and child:
                _walk(child)

    _walk(items)
    return flat


def audit_questionnaire_item(item: dict[str, Any]) -> list[str]:
    """Run structural checks for one Questionnaire item (recursive)."""
    issues: list[str] = []

    item_type = item.get("type")
    if not isinstance(item_type, str) or item_type not in FHIR_ALLOWED_TYPES:
        issues.append("invalid_item_type")

    link_id = item.get("linkId")
    if not isinstance(link_id, str) or not link_id.strip():
        issues.append("invalid_item_linkId")

    text = item.get("text")
    if not isinstance(text, str) or not text.strip():
        issues.append("missing_item_text")

    if "required" in item and not isinstance(item.get("required"), bool):
        issues.append("invalid_item_required")

    if "code" in item and not is_valid_code_array(item.get("code")):
        issues.append("invalid_item_code")

    if item_type in {"choice", "open-choice"} and not has_valid_answer_option(item):
        issues.append("invalid_item_answerOption")

    children = item.get("item")
    if children is not None and not isinstance(children, list):
        issues.append("invalid_item_children")

    if isinstance(children, list):
        for child in children:
            if not isinstance(child, dict):
                issues.append("invalid_item_children")
                continue
            issues.extend(audit_questionnaire_item(child))

    return issues


def audit_questionnaire(payload: dict[str, Any]) -> list[str]:
    """Run Questionnaire-level structural checks."""
    issues: list[str] = []

    if payload.get("resourceType") != "Questionnaire":
        issues.append("invalid_resourceType")

    for required_str_field in ("id", "status", "title"):
        value = payload.get(required_str_field)
        if not isinstance(value, str) or not value.strip():
            issues.append(f"missing_or_invalid_{required_str_field}")

    root_items = payload.get("item")
    if not isinstance(root_items, list) or len(root_items) == 0:
        issues.append("missing_root_items")
        return issues

    for root_item in root_items:
        if not isinstance(root_item, dict):
            issues.append("invalid_root_item")
            continue
        issues.extend(audit_questionnaire_item(root_item))

    return issues


def run_fhir_audit(records: list[RecordParseResult]) -> AuditSummary:
    """Audit all parsed records and produce dataset-level statistics."""
    type_counter: Counter[str] = Counter()
    issue_counter: Counter[str] = Counter()

    valid_records = 0
    total_items = 0

    for rec in records:
        payload = rec.assistant_obj
        all_items = iter_questionnaire_items(payload.get("item"))
        total_items += len(all_items)

        for item in all_items:
            item_type = item.get("type")
            if isinstance(item_type, str):
                type_counter[item_type] += 1
            else:
                type_counter["<missing-or-invalid>"] += 1

        issues = audit_questionnaire(payload)
        if len(issues) == 0:
            valid_records += 1
        else:
            issue_counter.update(issues)

    total_records = len(records)
    invalid_records = total_records - valid_records
    pass_rate = (valid_records / total_records * 100.0) if total_records else 0.0
    avg_items_per_form = (total_items / total_records) if total_records else 0.0

    return AuditSummary(
        total_records=total_records,
        valid_records=valid_records,
        invalid_records=invalid_records,
        pass_rate=pass_rate,
        type_counts=dict(sorted(type_counter.items(), key=lambda x: x[0])),
        issue_counts=dict(sorted(issue_counter.items(), key=lambda x: (-x[1], x[0]))),
        avg_items_per_form=avg_items_per_form,
    )


audit_summary = run_fhir_audit(parsed_records)

print("FHIR Questionnaire Audit Summary")
print("-" * 60)
print(f"Total records:       {audit_summary.total_records}")
print(f"Valid records:       {audit_summary.valid_records}")
print(f"Invalid records:     {audit_summary.invalid_records}")
print(f"Pass rate:           {audit_summary.pass_rate:.2f}%")
print(f"Avg items per form:  {audit_summary.avg_items_per_form:.2f}")

print("\nItem type distribution:")
for t, count in audit_summary.type_counts.items():
    print(f"- {t}: {count}")

print("\nIssue counts:")
if audit_summary.issue_counts:
    for issue, count in audit_summary.issue_counts.items():
        print(f"- {issue}: {count}")
else:
    print("- none")

FHIR Questionnaire Audit Summary
------------------------------------------------------------
Total records:       1500
Valid records:       1500
Invalid records:     0
Pass rate:           100.00%
Avg items per form:  5.80

Item type distribution:
- boolean: 1350
- choice: 3300
- decimal: 450
- integer: 3450
- string: 150

Issue counts:
- none


## Training Configuration and Rationale

### Objective

We fine-tune **the selected base instruct model** to generate **complete FHIR R4 Questionnaire JSON** from instruction prompts, prioritizing:

- questionnaire-level structural validity,
- coherent multi-item form composition,
- robust behavior across mixed item types and nested groups.

### Adaptation Strategy

We use a parameter-efficient setup (LoRA/QLoRA-style) to reduce memory footprint and improve practical reproducibility on Apple Silicon.

Why this choice:

- lower compute requirements than full fine-tuning,
- faster iteration cycles for ablation and error analysis,
- easier artifact sharing (adapter weights instead of full model checkpoints).

### Core Hyperparameter Dimensions

The experiment tracks and justifies:

1. model identifier and tokenizer pairing,
2. LoRA rank and scaling parameters,
3. sequence length and effective batch size,
4. optimizer and learning rate schedule,
5. number of training steps / iterations,
6. validation interval and checkpoint cadence.

### Reproducibility Controls

This training section is tied to:

- fixed global and split seeds,
- explicit train/validation manifest,
- persisted runtime metadata,
- versioned configuration snapshot.

### Reporting Policy

For academic transparency, we will report:

- final hyperparameter configuration,
- training dynamics (loss curve, runtime),
- validation metrics and FHIR pass-rate,
- representative qualitative generations.

In [5]:
# Reproducible train/validation split (stratified by dominant Questionnaire item type)

from __future__ import annotations

import json
import random
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

SPLIT_SEED = 173
VAL_RATIO = 0.20


def iter_questionnaire_items(items: Any) -> list[dict[str, Any]]:
    """Flatten questionnaire items recursively."""
    flat: list[dict[str, Any]] = []

    def _walk(nodes: Any) -> None:
        """Recursively traverse nested item arrays and collect item nodes."""
        if not isinstance(nodes, list):
            return
        for node in nodes:
            if not isinstance(node, dict):
                continue
            flat.append(node)
            children = node.get("item")
            if isinstance(children, list) and children:
                _walk(children)

    _walk(items)
    return flat


def get_primary_item_type(payload: dict[str, Any]) -> str:
    """Return dominant item type in a Questionnaire for split stratification."""
    type_counter: Counter[str] = Counter()
    for item in iter_questionnaire_items(payload.get("item")):
        item_type = item.get("type")
        key = item_type if isinstance(item_type, str) and item_type.strip() else "<missing-or-invalid>"
        type_counter[key] += 1

    if not type_counter:
        return "<missing-or-invalid>"

    return sorted(type_counter.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]


def build_stratified_buckets(records: list[RecordParseResult]) -> dict[str, list[int]]:
    """Group record indices by dominant Questionnaire item type."""
    buckets: dict[str, list[int]] = defaultdict(list)
    for idx, rec in enumerate(records):
        buckets[get_primary_item_type(rec.assistant_obj)].append(idx)
    return dict(buckets)


def stratified_train_val_split(
    records: list[RecordParseResult],
    val_ratio: float,
    seed: int,
) -> tuple[list[int], list[int]]:
    """Create deterministic train/validation indices with type stratification."""
    if not (0.0 < val_ratio < 1.0):
        raise ValueError("val_ratio must be between 0 and 1")
    if len(records) < 2:
        raise ValueError("At least two records are required for splitting")

    rng = random.Random(seed)
    buckets = build_stratified_buckets(records)

    train_idx: list[int] = []
    val_idx: list[int] = []

    for _, indices in buckets.items():
        shuffled = indices[:]
        rng.shuffle(shuffled)

        n_total = len(shuffled)
        n_val = max(1, int(round(n_total * val_ratio))) if n_total > 1 else 0
        n_val = min(n_val, n_total - 1) if n_total > 1 else 0

        val_idx.extend(shuffled[:n_val])
        train_idx.extend(shuffled[n_val:])

    train_idx.sort()
    val_idx.sort()

    if set(train_idx).intersection(val_idx):
        raise RuntimeError("Train/validation overlap detected")
    if len(train_idx) + len(val_idx) != len(records):
        raise RuntimeError("Split size mismatch")

    return train_idx, val_idx


def summarize_type_distribution(records: list[RecordParseResult], indices: list[int]) -> dict[str, int]:
    """Summarize dominant Questionnaire type counts for a subset."""
    counter: Counter[str] = Counter()
    for idx in indices:
        counter[get_primary_item_type(records[idx].assistant_obj)] += 1
    return dict(sorted(counter.items(), key=lambda x: x[0]))


def save_split_manifest(
    output_path: Path,
    train_idx: list[int],
    val_idx: list[int],
    seed: int,
    val_ratio: float,
    train_dist: dict[str, int],
    val_dist: dict[str, int],
) -> None:
    """Persist split metadata for reproducibility and auditability."""
    payload = {
        "seed": seed,
        "val_ratio": val_ratio,
        "n_train": len(train_idx),
        "n_val": len(val_idx),
        "train_indices": train_idx,
        "val_indices": val_idx,
        "train_type_distribution": train_dist,
        "val_type_distribution": val_dist,
    }
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


train_indices, val_indices = stratified_train_val_split(
    records=parsed_records,
    val_ratio=VAL_RATIO,
    seed=SPLIT_SEED,
)

train_distribution = summarize_type_distribution(parsed_records, train_indices)
val_distribution = summarize_type_distribution(parsed_records, val_indices)

split_manifest_path = repo_root / "lab" / "artifacts" / "split_manifest.json"
save_split_manifest(
    output_path=split_manifest_path,
    train_idx=train_indices,
    val_idx=val_indices,
    seed=SPLIT_SEED,
    val_ratio=VAL_RATIO,
    train_dist=train_distribution,
    val_dist=val_distribution,
)

print("Split summary")
print("-" * 60)
print(f"Total records: {len(parsed_records)}")
print(f"Train size:    {len(train_indices)}")
print(f"Val size:      {len(val_indices)}")
print(f"Seed:          {SPLIT_SEED}")
print(f"Manifest:      {split_manifest_path}")

print("\nTrain dominant type distribution:")
for t, c in train_distribution.items():
    print(f"- {t}: {c}")

print("\nValidation dominant type distribution:")
for t, c in val_distribution.items():
    print(f"- {t}: {c}")

Split summary
------------------------------------------------------------
Total records: 1500
Train size:    1200
Val size:      300
Seed:          173
Manifest:      /Users/CAE9/aiprom-llm/lab/artifacts/split_manifest.json

Train dominant type distribution:
- boolean: 360
- choice: 360
- integer: 480

Validation dominant type distribution:
- boolean: 90
- choice: 90
- integer: 120


In [6]:
# Training configuration: canonical object + persistence for reproducibility

from __future__ import annotations

import json
from datetime import UTC, datetime
from pathlib import Path
from typing import Any


def build_training_config(
    model_name: str,
    train_size: int,
    val_size: int,
    seed: int,
) -> dict[str, Any]:
    """Build a reproducible training configuration dictionary."""
    artifact_root = Path(artifacts_dir) if "artifacts_dir" in globals() else (repo_root / "lab" / "artifacts")
    split_manifest = Path(split_manifest_path) if "split_manifest_path" in globals() else (artifact_root / "split_manifest.json")

    return {
        "experiment_name": f"{MODEL_ALIAS}_fhir_questionnaire_lora" if "MODEL_ALIAS" in globals() else "fhir_questionnaire_lora",
        "created_utc": datetime.now(UTC).isoformat(),
        "model": {
            "backend": MODEL_BACKEND if "MODEL_BACKEND" in globals() else "mlx_lm",
            "name": model_name,
            "task": "chatml_to_fhir_questionnaire_json",
        },
        "data": {
            "dataset_path": str(dataset_path),
            "n_train": train_size,
            "n_val": val_size,
            "split_manifest_path": str(split_manifest),
        },
        "reproducibility": {
            "global_seed": seed,
            "split_seed": SPLIT_SEED,
            "python_version": runtime_info.get("python_version"),
            "platform": runtime_info.get("platform"),
            "git_commit": runtime_info.get("git_commit"),
        },
        "training": {
            "strategy": "lora_or_qlora",
            "max_seq_len": 2048,
            "batch_size": 2,
            "grad_accum_steps": 8,
            "learning_rate": 3e-5,
            "weight_decay": 0.0,
            "warmup_steps": 30,
            "train_steps": 800,
            "eval_interval": 50,
            "save_interval": 50,
            "optimizer": "adamw",
            "mask_prompt": False,
            "num_layers": 12,
        },
        "lora": {
            "rank": 16,
            "alpha": 32,
            "dropout": 0.05,
            "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
        },
        "outputs": {
            "artifact_dir": str(artifact_root),
            "checkpoint_dir": str(artifact_root / "checkpoints_stable"),
            "log_json_path": str(artifact_root / "training_config_stable.json"),
        },
    }


def persist_training_config(config: dict[str, Any], output_path: Path) -> None:
    """Write training configuration to disk in JSON format."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(config, indent=2), encoding="utf-8")


def print_training_config_summary(config: dict[str, Any]) -> None:
    """Print a compact summary of the current training setup."""
    print("Training configuration summary")
    print("-" * 60)
    print(f"Experiment:     {config['experiment_name']}")
    print(f"Model backend:  {config['model'].get('backend', 'mlx_lm')}")
    print(f"Model:          {config['model']['name']}")
    print(f"Task:           {config['model']['task']}")
    print(f"Train/Val:      {config['data']['n_train']} / {config['data']['n_val']}")
    print(f"Seed:           {config['reproducibility']['global_seed']}")
    print(f"Train steps:    {config['training']['train_steps']}")
    print(f"Learning rate:  {config['training']['learning_rate']}")
    print(f"Batch size:     {config['training']['batch_size']}")
    print(f"Grad accum:     {config['training']['grad_accum_steps']}")
    print(f"Seq length:     {config['training']['max_seq_len']}")
    print(f"Config JSON:    {config['outputs']['log_json_path']}")


SELECTED_MODEL_NAME = str(globals().get("MODEL_NAME", "org/model-instruct")).strip()

training_config = build_training_config(
    model_name=SELECTED_MODEL_NAME,
    train_size=len(train_indices),
    val_size=len(val_indices),
    seed=GLOBAL_SEED,
)

config_path = Path(training_config["outputs"]["log_json_path"])
persist_training_config(training_config, config_path)
print_training_config_summary(training_config)

Training configuration summary
------------------------------------------------------------
Experiment:     mlx-community-qwen2.5-3b-instruct-4bit_fhir_questionnaire_lora
Model backend:  mlx_lm
Model:          mlx-community/Qwen2.5-3B-Instruct-4bit
Task:           chatml_to_fhir_questionnaire_json
Train/Val:      1200 / 300
Seed:           173
Train steps:    800
Learning rate:  3e-05
Batch size:     2
Grad accum:     8
Seq length:     2048
Config JSON:    /Users/CAE9/aiprom-llm/lab/artifacts/mlx-community-qwen2.5-3b-instruct-4bit/training_config_stable.json


## Data Materialization for Training

This section converts parsed prompt/completion records and deterministic split indices into train/validation files ready for MLX-LM LoRA training.

### Goals

- Produce reproducible train/validation JSONL artifacts.
- Store ChatML transcripts derived from prompt/completion rows.
- Persist metadata for traceability and later audit.

### Output Artifacts

- `lab/artifacts/train.jsonl`
- `lab/artifacts/val.jsonl`
- `lab/artifacts/dataset_manifest.json`

These files are generated deterministically from the parsed synthetic dataset and split manifest.

In [7]:
# Build deterministic train/validation files for MLX-LM

from __future__ import annotations

import json
from pathlib import Path
from typing import Any


def build_chatml_record(text: str) -> dict[str, str]:
    """Create a minimal MLX-LM-compatible JSONL record from ChatML text.

    Parameters
    ----------
    text:
        ChatML transcript string containing system/user/assistant turns.

    Returns
    -------
    dict[str, str]
        Single record with one `text` field.
    """
    return {"text": text}


def materialize_split_jsonl(
    records: list[RecordParseResult],
    indices: list[int],
    output_path: Path,
) -> None:
    """Write selected records into JSONL format.

    Parameters
    ----------
    records:
        Parsed dataset records.
    indices:
        Record indices to include in the output file.
    output_path:
        Destination JSONL path.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        for idx in indices:
            row = build_chatml_record(records[idx].text)
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def write_dataset_manifest(
    manifest_path: Path,
    train_path: Path,
    val_path: Path,
    valid_path: Path,
    train_size: int,
    val_size: int,
) -> None:
    """Persist dataset materialization metadata for reproducibility.

    Parameters
    ----------
    manifest_path:
        Output path for JSON manifest.
    train_path:
        Train JSONL artifact path.
    val_path:
        Validation JSONL artifact path.
    valid_path:
        Validation JSONL alias path for tool compatibility.
    train_size:
        Number of train records.
    val_size:
        Number of validation records.
    """
    payload = {
        "dataset_source": str(dataset_path),
        "train_path": str(train_path),
        "val_path": str(val_path),
        "valid_path": str(valid_path),
        "n_train": train_size,
        "n_val": val_size,
        "split_seed": SPLIT_SEED,
        "global_seed": GLOBAL_SEED,
    }
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


artifacts_dir = repo_root / "lab" / "artifacts"
train_jsonl_path = artifacts_dir / "train.jsonl"
val_jsonl_path = artifacts_dir / "val.jsonl"
valid_jsonl_path = artifacts_dir / "valid.jsonl"
dataset_manifest_path = artifacts_dir / "dataset_manifest.json"

materialize_split_jsonl(parsed_records, train_indices, train_jsonl_path)
materialize_split_jsonl(parsed_records, val_indices, val_jsonl_path)
materialize_split_jsonl(parsed_records, val_indices, valid_jsonl_path)
write_dataset_manifest(
    dataset_manifest_path,
    train_jsonl_path,
    val_jsonl_path,
    valid_jsonl_path,
    len(train_indices),
    len(val_indices),
)

print("Materialization complete")
print("-" * 60)
print(f"Train JSONL: {train_jsonl_path}")
print(f"Val JSONL:   {val_jsonl_path}")
print(f"Valid JSONL: {valid_jsonl_path}")
print(f"Manifest:    {dataset_manifest_path}")
print(f"Train rows:  {len(train_indices)}")
print(f"Val rows:    {len(val_indices)}")

Materialization complete
------------------------------------------------------------
Train JSONL: /Users/CAE9/aiprom-llm/lab/artifacts/train.jsonl
Val JSONL:   /Users/CAE9/aiprom-llm/lab/artifacts/val.jsonl
Valid JSONL: /Users/CAE9/aiprom-llm/lab/artifacts/valid.jsonl
Manifest:    /Users/CAE9/aiprom-llm/lab/artifacts/dataset_manifest.json
Train rows:  1200
Val rows:    300


## Training Execution (MLX-LM)

This section launches LoRA fine-tuning using MLX-LM with the persisted configuration.

### Notes

- The command is generated programmatically from `training_config`.
- If the training command fails due to environment-specific CLI differences, the notebook reports stderr and keeps reproducibility artifacts.
- Output checkpoints are written under the selected model artifact directory (`lab/artifacts/<MODEL_ALIAS>/checkpoints_stable`).

## Energy Accounting and Carbon Estimation

This section instruments command-level energy and emissions accounting for key workflow steps such as training and inference.

### What this section does

- installs and configures `codecarbon` when needed,
- provides a reusable wrapper to execute shell commands while tracking energy and CO₂,
- writes reproducible artifacts under `lab/artifacts/energy/`.

### Output artifacts

- `emissions.csv`: consolidated CodeCarbon log,
- `*_energy.json`: per-step summaries including duration, return code, kWh, and kgCO2eq.

### Scope note

Energy and emissions are estimates based on hardware utilization and offline country factors; they are suitable for comparative reporting across runs.

In [8]:
# Build deterministic train/validation files for MLX-LM

from __future__ import annotations

import json
from pathlib import Path
from typing import Any


def build_chatml_record(text: str) -> dict[str, str]:
    """Create a minimal MLX-LM-compatible JSONL record from ChatML text.

    Parameters
    ----------
    text:
        ChatML transcript string containing system/user/assistant turns.

    Returns
    -------
    dict[str, str]
        Single record with one `text` field.
    """
    return {"text": text}


def materialize_split_jsonl(
    records: list[RecordParseResult],
    indices: list[int],
    output_path: Path,
) -> None:
    """Write selected records into JSONL format.

    Parameters
    ----------
    records:
        Parsed dataset records.
    indices:
        Record indices to include in the output file.
    output_path:
        Destination JSONL path.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        for idx in indices:
            row = build_chatml_record(records[idx].text)
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def write_dataset_manifest(
    manifest_path: Path,
    train_path: Path,
    val_path: Path,
    valid_path: Path,
    train_size: int,
    val_size: int,
) -> None:
    """Persist dataset materialization metadata for reproducibility.

    Parameters
    ----------
    manifest_path:
        Output path for JSON manifest.
    train_path:
        Path to train JSONL.
    val_path:
        Path to validation JSONL.
    valid_path:
        Path to duplicate validation JSONL expected by some MLX setups.
    train_size:
        Number of records written to train.
    val_size:
        Number of records written to validation.
    """
    payload: dict[str, Any] = {
        "train_path": str(train_path),
        "val_path": str(val_path),
        "valid_path": str(valid_path),
        "train_size": train_size,
        "val_size": val_size,
        "dataset_path": str(dataset_path),
        "split_manifest_path": str(split_manifest_path),
        "split_seed": SPLIT_SEED,
        "global_seed": GLOBAL_SEED,
    }
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


artifact_root = Path(artifacts_dir) if "artifacts_dir" in globals() else (repo_root / "lab" / "artifacts")
artifact_root.mkdir(parents=True, exist_ok=True)

train_jsonl_path = artifact_root / "train.jsonl"
val_jsonl_path = artifact_root / "val.jsonl"
valid_jsonl_path = artifact_root / "valid.jsonl"
dataset_manifest_path = artifact_root / "dataset_manifest.json"

materialize_split_jsonl(parsed_records, train_indices, train_jsonl_path)
materialize_split_jsonl(parsed_records, val_indices, val_jsonl_path)
materialize_split_jsonl(parsed_records, val_indices, valid_jsonl_path)
write_dataset_manifest(
    dataset_manifest_path,
    train_jsonl_path,
    val_jsonl_path,
    valid_jsonl_path,
    len(train_indices),
    len(val_indices),
)

print("Materialization complete")
print("-" * 60)
print(f"Artifact root: {artifact_root}")
print(f"Train JSONL:   {train_jsonl_path}")
print(f"Val JSONL:     {val_jsonl_path}")
print(f"Valid JSONL:   {valid_jsonl_path}")
print(f"Manifest:      {dataset_manifest_path}")
print(f"Train rows:    {len(train_indices)}")
print(f"Val rows:      {len(val_indices)}")

Materialization complete
------------------------------------------------------------
Artifact root: /Users/CAE9/aiprom-llm/lab/artifacts
Train JSONL:   /Users/CAE9/aiprom-llm/lab/artifacts/train.jsonl
Val JSONL:     /Users/CAE9/aiprom-llm/lab/artifacts/val.jsonl
Valid JSONL:   /Users/CAE9/aiprom-llm/lab/artifacts/valid.jsonl
Manifest:      /Users/CAE9/aiprom-llm/lab/artifacts/dataset_manifest.json
Train rows:    1200
Val rows:      300


In [9]:
# Construct and execute MLX-LM training command

from __future__ import annotations

import json
import shlex
import subprocess
from pathlib import Path
from typing import Any

RUN_TRAINING = True


def get_mlx_lm_lora_help(cwd: Path) -> str:
    """Return help text for `python -m mlx_lm lora`."""
    completed = subprocess.run(
        ["python", "-m", "mlx_lm", "lora", "--help"],
        cwd=str(cwd),
        text=True,
        capture_output=True,
    )
    return (completed.stdout or "") + "\n" + (completed.stderr or "")


def detect_dataset_format(data_dir: Path) -> str:
    """Best-effort detection of MLX-LM local dataset format.

    Returns: 'text' | 'messages' | 'prompt_completion' | 'unknown'
    """
    candidate_files = ["train.jsonl", "valid.jsonl", "val.jsonl", "test.jsonl"]
    dataset_file = next((data_dir / name for name in candidate_files if (data_dir / name).exists()), None)
    if dataset_file is None:
        return "unknown"

    with dataset_file.open("r", encoding="utf-8") as f:
        first_line = f.readline().strip()

    if not first_line:
        return "unknown"

    try:
        record = json.loads(first_line)
    except json.JSONDecodeError:
        return "unknown"

    if isinstance(record, dict):
        if "text" in record:
            return "text"
        if "messages" in record:
            return "messages"
        if "prompt" in record and ("completion" in record or "response" in record):
            return "prompt_completion"

    return "unknown"


def build_mlx_lm_lora_command(
    config: dict[str, Any],
    help_text: str,
    data_dir: Path,
) -> tuple[list[str], list[str]]:
    """Build MLX-LM LoRA command line arguments from config.

    Returns (command, warnings).
    """
    training = config["training"]
    outputs = config["outputs"]
    model_name = str(config["model"]["name"]).strip()

    if not model_name.startswith("mlx-community/"):
        raise ValueError(
            f"Invalid base model for this notebook: {model_name}. "
            "Use a public mlx-community model to avoid auth failures."
        )

    dataset_format = detect_dataset_format(data_dir)
    warnings: list[str] = []

    command = [
        "python",
        "-m",
        "mlx_lm",
        "lora",
        "--model",
        model_name,
        "--train",
        "--data",
        str(data_dir),
        "--iters",
        str(training["train_steps"]),
        "--batch-size",
        str(training["batch_size"]),
        "--learning-rate",
        str(training["learning_rate"]),
        "--steps-per-report",
        str(training["eval_interval"]),
        "--steps-per-eval",
        str(training["eval_interval"]),
        "--save-every",
        str(training["save_interval"]),
        "--adapter-path",
        str(Path(outputs["checkpoint_dir"])),
    ]

    if "--grad-accumulation-steps" in help_text:
        command.extend(["--grad-accumulation-steps", str(training["grad_accum_steps"])])
    if "--max-seq-length" in help_text:
        command.extend(["--max-seq-length", str(training["max_seq_len"])])
    if "--optimizer" in help_text and training.get("optimizer"):
        command.extend(["--optimizer", str(training["optimizer"])])

    if bool(training.get("mask_prompt", False)) and dataset_format == "text":
        warnings.append(
            "mask_prompt=true pero el dataset es formato text; omitiendo --mask-prompt "
            "(MLX-LM no soporta prompt masking para text dataset)."
        )
    elif "--mask-prompt" in help_text and bool(training.get("mask_prompt", False)):
        command.append("--mask-prompt")

    if "--num-layers" in help_text and training.get("num_layers") is not None:
        command.extend(["--num-layers", str(training["num_layers"])])
    if "--seed" in help_text:
        command.extend(["--seed", str(config["reproducibility"]["global_seed"])])

    return command, warnings


def run_command(command: list[str], cwd: Path) -> tuple[int, str, str]:
    """Run a subprocess command and capture stdout/stderr."""
    completed = subprocess.run(
        command,
        cwd=str(cwd),
        text=True,
        capture_output=True,
    )
    return completed.returncode, completed.stdout, completed.stderr


def persist_run_log(log_path: Path, payload: dict[str, Any]) -> None:
    """Write training run log payload to JSON."""
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def adapter_artifacts_exist(checkpoint_dir: Path) -> bool:
    """Return True if adapter files exist in checkpoint directory tree."""
    markers = {"adapters.safetensors", "adapter_model.safetensors"}
    if not checkpoint_dir.exists():
        return False
    for file in checkpoint_dir.rglob("*"):
        if file.is_file() and file.name in markers:
            return True
    return False


def replace_arg(command: list[str], arg_name: str, new_value: str) -> list[str]:
    """Replace one CLI argument value in a command list."""
    cmd = command[:]
    if arg_name in cmd:
        idx = cmd.index(arg_name)
        if idx + 1 < len(cmd):
            cmd[idx + 1] = str(new_value)
    return cmd


def build_stability_retry_command(base_command: list[str]) -> list[str]:
    """Return a conservative command profile for Metal stability."""
    cmd = base_command[:]
    cmd = replace_arg(cmd, "--batch-size", "1")
    cmd = replace_arg(cmd, "--max-seq-length", "1024")
    cmd = replace_arg(cmd, "--grad-accumulation-steps", "16")
    cmd = replace_arg(cmd, "--num-layers", "8")
    cmd = replace_arg(cmd, "--steps-per-eval", "100")
    cmd = replace_arg(cmd, "--steps-per-report", "100")
    cmd = replace_arg(cmd, "--save-every", "100")
    return cmd


def is_metal_runtime_error(stderr_text: str) -> bool:
    """Detect common Apple Metal runtime crashes during MLX training."""
    lowered = (stderr_text or "").lower()
    return (
        "[metal]" in lowered
        or "impacting interactivity" in lowered
        or "kiogpucommandbuffercallbackerrorimpactinginteractivity" in lowered
        or "command buffer execution failed" in lowered
    )


help_text = get_mlx_lm_lora_help(repo_root)
train_command, warnings = build_mlx_lm_lora_command(training_config, help_text, artifacts_dir)
train_command_str = " ".join(shlex.quote(x) for x in train_command)
checkpoint_dir = Path(training_config["outputs"]["checkpoint_dir"])

for w in warnings:
    print(f"WARNING: {w}")

print("Training command")
print("-" * 60)
print(train_command_str)

if not RUN_TRAINING:
    print("\nRUN_TRAINING is False: skipping execution (command validated only).")
else:
    attempts: list[dict[str, Any]] = []

    return_code, stdout, stderr = run_command(train_command, repo_root)
    attempts.append(
        {
            "name": "primary",
            "command": train_command,
            "return_code": return_code,
            "stdout_preview": stdout[-4000:],
            "stderr_preview": stderr[-4000:],
        }
    )

    if return_code != 0 and is_metal_runtime_error(stderr):
        print("\nDetected Metal runtime failure. Retrying with a conservative stability profile...")
        retry_command = build_stability_retry_command(train_command)
        print("Retry command")
        print("-" * 60)
        print(" ".join(shlex.quote(x) for x in retry_command))

        return_code, stdout, stderr = run_command(retry_command, repo_root)
        attempts.append(
            {
                "name": "metal_stability_retry",
                "command": retry_command,
                "return_code": return_code,
                "stdout_preview": stdout[-4000:],
                "stderr_preview": stderr[-4000:],
            }
        )

    training_run_log = {
        "attempts": attempts,
        "final_return_code": return_code,
    }

    training_run_log_path = artifacts_dir / "training_run_log.json"
    persist_run_log(training_run_log_path, training_run_log)

    print("\nTraining execution result")
    print("-" * 60)
    print(f"Return code: {return_code}")
    print(f"Run log:     {training_run_log_path}")
    print(f"Checkpoint:  {checkpoint_dir}")

    if return_code != 0:
        print("\nTraining command returned non-zero exit code.")
        print("Last stderr lines:")
        print(stderr[-1200:])
        raise RuntimeError("LoRA training failed; adapter was not generated.")

    if not adapter_artifacts_exist(checkpoint_dir):
        raise RuntimeError(
            "Training finished but no adapter artifacts were found. "
            "Expected adapters.safetensors/adapter_model.safetensors under checkpoint_dir."
        )

    print("Training finished successfully and adapter artifacts were detected.")

Training command
------------------------------------------------------------
python -m mlx_lm lora --model mlx-community/Qwen2.5-3B-Instruct-4bit --train --data /Users/CAE9/aiprom-llm/lab/artifacts --iters 800 --batch-size 2 --learning-rate 3e-05 --steps-per-report 50 --steps-per-eval 50 --save-every 50 --adapter-path /Users/CAE9/aiprom-llm/lab/artifacts/mlx-community-qwen2.5-3b-instruct-4bit/checkpoints_stable --grad-accumulation-steps 8 --max-seq-length 2048 --optimizer adamw --num-layers 12 --seed 173

Training execution result
------------------------------------------------------------
Return code: 0
Run log:     /Users/CAE9/aiprom-llm/lab/artifacts/training_run_log.json
Checkpoint:  /Users/CAE9/aiprom-llm/lab/artifacts/mlx-community-qwen2.5-3b-instruct-4bit/checkpoints_stable
Training finished successfully and adapter artifacts were detected.


## GGUF Deliverable for LM Studio

This section fuses the trained LoRA adapter into the base model and exports a GGUF artifact for LM Studio.

### Purpose

- Produce a single portable GGUF model file.
- Avoid loading LoRA separately inside LM Studio.
- Keep export metadata reproducible in `lab/artifacts/`.

Set `RUN_GGUF_EXPORT = True` to execute the export command.

In [ ]:
# Fuse LoRA adapter and export GGUF for LM Studio

from __future__ import annotations

import json
import shlex
import subprocess
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

RUN_GGUF_EXPORT = False


def resolve_repo_root() -> Path:
    """Best-effort repo root resolver for notebook sessions."""
    if "repo_root" in globals():
        try:
            candidate = Path(repo_root)
            if candidate.exists():
                return candidate
        except Exception:
            pass

    cwd = Path.cwd().resolve()
    for parent in [cwd, *cwd.parents]:
        if (parent / "requirements.txt").exists() and (parent / "lab").exists():
            return parent

    return cwd


ROOT_DIR = resolve_repo_root()
ARTIFACTS_DIR = Path(artifacts_dir) if "artifacts_dir" in globals() else (ROOT_DIR / "lab" / "artifacts")
DEFAULT_PUBLIC_MODEL = "mlx-community/Qwen2.5-3B-Instruct-4bit"


def slugify_model_name(model_name: str) -> str:
    """Convert a model name into the notebook's artifact directory convention."""
    return (
        (model_name or "")
        .strip()
        .lower()
        .replace("/", "-")
        .replace(" ", "-")
    )


def adapter_artifacts_exist(checkpoint_dir: Path) -> bool:
    """Return True if adapter files exist in checkpoint directory tree."""
    markers = {"adapters.safetensors", "adapter_model.safetensors"}
    if not checkpoint_dir.exists() or not checkpoint_dir.is_dir():
        return False

    if any((checkpoint_dir / marker).exists() for marker in markers):
        return True

    for file in checkpoint_dir.rglob("*"):
        if file.is_file() and file.name in markers:
            return True

    return False


def candidate_adapter_dirs(artifacts_root: Path, model_name: str = "") -> list[Path]:
    """Return likely adapter/checkpoint directories generated by MLX-LM LoRA training."""
    candidates: list[Path] = [
        artifacts_root / "adapters",
        artifacts_root / "checkpoints_stable",
        artifacts_root / "checkpoints",
        artifacts_root / "checkpoint",
    ]

    if model_name:
        alias = slugify_model_name(model_name)
        candidates.extend(
            [
                artifacts_root / alias / "checkpoints_stable",
                artifacts_root / alias / "adapters",
                artifacts_root / alias / "checkpoints",
            ]
        )

    # One-level deep scan: lab/artifacts/<MODEL_ALIAS>/checkpoints_stable
    try:
        for child in artifacts_root.iterdir():
            if not child.is_dir():
                continue
            candidates.extend(
                [
                    child / "checkpoints_stable",
                    child / "adapters",
                    child / "checkpoints",
                ]
            )
    except FileNotFoundError:
        pass

    # De-dup while preserving order
    seen: set[Path] = set()
    unique: list[Path] = []
    for p in candidates:
        if p in seen:
            continue
        seen.add(p)
        unique.append(p)
    return unique


def select_adapter_path(artifacts_root: Path, model_name: str = "") -> Path | None:
    """Select adapter directory if any candidate contains adapter artifacts."""
    # Prefer the exact checkpoint_dir from training_config when available.
    if "training_config" in globals() and isinstance(training_config, dict):
        try:
            ck = Path(training_config["outputs"]["checkpoint_dir"])
            if adapter_artifacts_exist(ck):
                return ck
        except Exception:
            pass

    for path in candidate_adapter_dirs(artifacts_root, model_name=model_name):
        if adapter_artifacts_exist(path):
            return path

    return None


def load_training_config_from_artifact_root(artifact_root: Path) -> dict[str, Any] | None:
    config_path = artifact_root / "training_config_stable.json"
    if not config_path.exists():
        return None
    try:
        return json.loads(config_path.read_text(encoding="utf-8"))
    except Exception:
        return None


def resolve_model_name(adapter_path: Path | None) -> str:
    """Resolve base model name: training_config -> training_config_stable.json -> fallback."""
    if "training_config" in globals() and isinstance(training_config, dict):
        name = str(training_config.get("model", {}).get("name", "")).strip()
        if name.startswith("mlx-community/"):
            return name

    if adapter_path is not None:
        artifact_root = adapter_path.parent
        config = load_training_config_from_artifact_root(artifact_root)
        if isinstance(config, dict):
            model = config.get("model")
            if isinstance(model, dict):
                name = str(model.get("name", "")).strip()
                if name.startswith("mlx-community/"):
                    return name

    return DEFAULT_PUBLIC_MODEL


def build_fuse_export_command(
    *,
    model_name: str,
    adapter_path: Path,
    fused_dir: Path,
    gguf_path: Path,
) -> list[str]:
    """Build MLX-LM command for fuse + GGUF export."""
    return [
        "python",
        "-m",
        "mlx_lm",
        "fuse",
        "--model",
        model_name,
        "--adapter-path",
        str(adapter_path),
        "--save-path",
        str(fused_dir),
        "--export-gguf",
        "--gguf-path",
        str(gguf_path),
    ]


def run_command(command: list[str], cwd: Path) -> tuple[int, str, str]:
    """Run command and return returncode/stdout/stderr."""
    completed = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
        check=False,
    )
    return completed.returncode, completed.stdout, completed.stderr


def save_gguf_export_log(log_path: Path, payload: dict[str, Any]) -> None:
    """Persist GGUF export run metadata."""
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


adapter_path = select_adapter_path(ARTIFACTS_DIR, model_name=resolve_model_name(None))
artifact_root = adapter_path.parent if adapter_path else ARTIFACTS_DIR
base_model_name = resolve_model_name(adapter_path)

fused_dir = artifact_root / "fused"
gguf_path = fused_dir / "model-fused.gguf"
log_path = artifact_root / "gguf_export_log.json"

print("GGUF export plan")
print("-" * 60)
print(f"Base model:     {base_model_name}")
print(f"Repo root:      {ROOT_DIR}")
print(f"Artifact root:  {artifact_root}")
print(f"Adapter path:   {adapter_path if adapter_path else '<not-found>'}")
print(f"Fused dir:      {fused_dir}")
print(f"GGUF path:      {gguf_path}")
print(f"Run enabled:    {RUN_GGUF_EXPORT}")

if not adapter_path:
    print("\nNo adapter artifacts found. Run training first, then re-run this cell.")
elif not RUN_GGUF_EXPORT:
    preview_command = build_fuse_export_command(
        model_name=base_model_name,
        adapter_path=adapter_path,
        fused_dir=fused_dir,
        gguf_path=gguf_path,
    )
    print("\nRUN_GGUF_EXPORT is False. Command preview:")
    print(shlex.join(preview_command))
else:
    fused_dir.mkdir(parents=True, exist_ok=True)

    fuse_command = build_fuse_export_command(
        model_name=base_model_name,
        adapter_path=adapter_path,
        fused_dir=fused_dir,
        gguf_path=gguf_path,
    )
    return_code, stdout_text, stderr_text = run_command(fuse_command, ROOT_DIR)

    payload = {
        "timestamp_utc": datetime.now(UTC).isoformat(),
        "command": fuse_command,
        "base_model": base_model_name,
        "adapter_path": str(adapter_path),
        "artifact_root": str(artifact_root),
        "fused_dir": str(fused_dir),
        "gguf_path": str(gguf_path),
        "return_code": return_code,
        "stdout": stdout_text,
        "stderr": stderr_text,
    }
    save_gguf_export_log(log_path, payload)

    print("\nGGUF export result")
    print("-" * 60)
    print(f"Return code: {return_code}")
    print(f"Log file:    {log_path}")
    print(f"GGUF exists: {gguf_path.exists()}")

    if return_code != 0:
        print("\nExport command failed. Last stderr chunk:")
        print(stderr_text[-1500:])
    else:
        print("GGUF deliverable generated successfully.")

payload if "payload" in locals() else None


GGUF export plan
------------------------------------------------------------
Base model:     org/model-instruct
Adapter path:   checkpoints_stable
Fused dir:      /Users/CAE9/aiprom-llm/lab/lab/artifacts/fused
GGUF path:      /Users/CAE9/aiprom-llm/lab/lab/artifacts/fused/model-fused.gguf
Run enabled:    False

RUN_GGUF_EXPORT is False. Command preview:
python -m mlx_lm fuse --model org/model-instruct --adapter-path checkpoints_stable --save-path /Users/CAE9/aiprom-llm/lab/lab/artifacts/fused --export-gguf --gguf-path /Users/CAE9/aiprom-llm/lab/lab/artifacts/fused/model-fused.gguf


## Adapter Inference Test

This section runs a quick inference sanity check using the trained LoRA adapter.

### Purpose

- Verify adapter loading works.
- Generate one FHIR-style response from a ChatML prompt.
- Persist a reproducible inference log artifact.

Set `RUN_INFERENCE_TEST = True` to execute generation.

In [7]:
# Quick inference test with trained adapter (strict: no fallback)

from __future__ import annotations

import json
import shlex
import subprocess
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

RUN_INFERENCE_TEST = True
INFERENCE_CHECKPOINT_ITERS = None  # None = use current active adapter (recommended after successful training)

INFERENCE_SEED = 173
INFERENCE_TEMP = 0.0
INFERENCE_TOP_P = 1.0
INFERENCE_MAX_TOKENS = 900


def resolve_repo_root() -> Path:
    """Best-effort repo root resolver for notebook sessions.

    Falls back to walking up from CWD until a directory contains both
    `requirements.txt` and a `lab/` folder.
    """
    if "repo_root" in globals():
        try:
            candidate = Path(repo_root)
            if candidate.exists():
                return candidate
        except Exception:
            pass

    cwd = Path.cwd().resolve()
    for parent in [cwd, *cwd.parents]:
        if (parent / "requirements.txt").exists() and (parent / "lab").exists():
            return parent

    return cwd


ROOT_DIR = resolve_repo_root()
ARTIFACTS_DIR = Path(artifacts_dir) if "artifacts_dir" in globals() else (ROOT_DIR / "lab" / "artifacts")
DEFAULT_PUBLIC_MODEL = "mlx-community/Qwen2.5-3B-Instruct-4bit"


def build_chatml_test_prompt() -> str:
    """Build a deterministic ChatML prompt for adapter inference sanity check."""
    return (
        "<|im_start|>system\n"
        "You are a FHIR R4 expert. Return ONLY raw JSON (no markdown, no code fences).\n"
        "Generate VALID FHIR R4 Questionnaire JSON only.\n"
        "Constraints: do NOT include `initial`; for choice items use `answerOption` as an array of objects, each with one `valueCoding` object containing `system`, `code`, `display`; do NOT nest sub-items under choice; keep it concise.\n"
        "<|im_end|>"
        "<|im_start|>user\n"
        "Generate a complete FHIR Questionnaire for a weekly recovery check-in form. "
        "Include resourceType='Questionnaire', id='weekly-recovery-checkin', status='active', title, and an item array with exactly 6 questions. "
        "Use mixed item types (choice, string, text, integer, boolean, date) and valid linkId values. "
        "For choice items include 4 answerOption values with valueCoding.display. Return only JSON.\n"
        "<|im_end|>"
        "<|im_start|>assistant\n"
    )


def resolve_inference_model_name() -> str:
    """Resolve model name robustly, preferring public mlx-community checkpoints."""
    configured = ""
    if "training_config" in globals() and isinstance(training_config, dict):
        configured = str(training_config.get("model", {}).get("name", "")).strip()

    model_from_globals = str(globals().get("MODEL_NAME", "")).strip()

    candidates = [configured, model_from_globals, DEFAULT_PUBLIC_MODEL]
    for candidate in candidates:
        if candidate.startswith("mlx-community/"):
            return candidate

    return DEFAULT_PUBLIC_MODEL


def load_training_config_from_artifact_root(artifact_root: Path) -> dict[str, Any] | None:
    """Load training_config_stable.json if present next to adapter artifacts."""
    config_path = artifact_root / "training_config_stable.json"
    if not config_path.exists():
        return None
    try:
        return json.loads(config_path.read_text(encoding="utf-8"))
    except Exception:
        return None


def resolve_model_name_from_artifact_root(artifact_root: Path, fallback: str) -> str:
    """Resolve base model name from saved training config when available."""
    config = load_training_config_from_artifact_root(artifact_root)
    if not isinstance(config, dict):
        return fallback
    model = config.get("model")
    if isinstance(model, dict):
        name = str(model.get("name", "")).strip()
        if name.startswith("mlx-community/"):
            return name
    return fallback


def slugify_model_name(model_name: str) -> str:
    """Convert a model name into the notebook's artifact directory convention."""
    return (
        (model_name or "")
        .strip()
        .lower()
        .replace("/", "-")
        .replace(" ", "-")
    )


def adapter_artifacts_exist(checkpoint_dir: Path) -> bool:
    """Return True if adapter files exist in checkpoint directory tree."""
    markers = {"adapters.safetensors", "adapter_model.safetensors"}
    if not checkpoint_dir.exists() or not checkpoint_dir.is_dir():
        return False

    if any((checkpoint_dir / marker).exists() for marker in markers):
        return True

    for file in checkpoint_dir.rglob("*"):
        if file.is_file() and file.name in markers:
            return True

    return False


def candidate_adapter_dirs(artifacts_root: Path, model_name: str = "") -> list[Path]:
    """Return likely adapter/checkpoint directories generated by MLX-LM LoRA training."""
    candidates: list[Path] = [
        artifacts_root / "checkpoints_stable",
        artifacts_root / "adapters",
        artifacts_root / "checkpoints",
        artifacts_root / "checkpoint",
    ]

    if model_name:
        alias = slugify_model_name(model_name)
        candidates.extend(
            [
                artifacts_root / alias / "checkpoints_stable",
                artifacts_root / alias / "adapters",
                artifacts_root / alias / "checkpoints",
            ]
        )

    # One-level deep scan: lab/artifacts/<MODEL_ALIAS>/checkpoints_stable
    try:
        for child in artifacts_root.iterdir():
            if not child.is_dir():
                continue
            candidates.extend(
                [
                    child / "checkpoints_stable",
                    child / "adapters",
                    child / "checkpoints",
                ]
            )
    except FileNotFoundError:
        pass

    # De-dup while preserving order
    seen: set[Path] = set()
    unique: list[Path] = []
    for p in candidates:
        if p in seen:
            continue
        seen.add(p)
        unique.append(p)
    return unique


def select_adapter_path(artifacts_root: Path, model_name: str = "") -> Path | None:
    """Find adapter directory with supported marker files."""
    # Prefer the exact checkpoint_dir from training_config when available.
    if "training_config" in globals() and isinstance(training_config, dict):
        try:
            ck = Path(training_config["outputs"]["checkpoint_dir"])
            if adapter_artifacts_exist(ck):
                return ck
        except Exception:
            pass

    for path in candidate_adapter_dirs(artifacts_root, model_name=model_name):
        if adapter_artifacts_exist(path):
            return path

    return None


def build_mlx_generate_command(
    model_name: str,
    prompt: str,
    *,
    adapter_path: Path,
    max_tokens: int,
    seed: int,
    temp: float,
    top_p: float,
) -> list[str]:
    """Build `mlx_lm generate` command with mandatory adapter path."""
    return [
        "python",
        "-m",
        "mlx_lm",
        "generate",
        "--model",
        model_name,
        "--ignore-chat-template",
        "--prompt",
        prompt,
        "--max-tokens",
        str(max_tokens),
        "--seed",
        str(seed),
        "--temp",
        str(temp),
        "--top-p",
        str(top_p),
        "--verbose",
        "F",
        "--extra-eos-token",
        "<|im_end|>",
        "--adapter-path",
        str(adapter_path),
    ]


def run_inference_command(command: list[str], cwd: Path) -> tuple[int, str, str]:
    """Run inference command and capture outputs."""
    process = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
        check=False,
    )
    return process.returncode, process.stdout, process.stderr


def select_checkpoint_file(adapter_dir: Path, iters: int | None) -> Path | None:
    """Pick a saved adapter snapshot file like 0000100_adapters.safetensors when available."""
    if iters is None:
        return None
    candidate = adapter_dir / f"{iters:07d}_adapters.safetensors"
    return candidate if candidate.exists() else None


def stage_checkpoint_as_active(adapter_dir: Path, checkpoint_file: Path) -> Path:
    """Copy a snapshot file onto adapters.safetensors so mlx_lm can load it via --adapter-path."""
    active = adapter_dir / "adapters.safetensors"
    backup = adapter_dir / "adapters.safetensors.bak"
    if active.exists() and not backup.exists():
        active.replace(backup)
    active.write_bytes(checkpoint_file.read_bytes())
    return active


def extract_generated_json(stdout: str) -> tuple[dict[str, Any] | None, str | None]:
    """Extract generated JSON object from model stdout when possible.

    Returns
    -------
    tuple[obj_or_none, parse_error_or_none]
    """
    text = (stdout or "").strip()
    if not text:
        return None, "STDOUT is empty"

    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            return parsed, None
    except Exception:
        pass

    fenced_start = text.find("```json")
    if fenced_start >= 0:
        start_idx = fenced_start + len("```json")
        fenced_end = text.find("```", start_idx)
        if fenced_end > start_idx:
            candidate = text[start_idx:fenced_end].strip()
            try:
                parsed = json.loads(candidate)
                if isinstance(parsed, dict):
                    return parsed, None
            except Exception as exc:
                return None, f"Fenced JSON parse error: {exc}"

    first_brace = text.find("{")
    if first_brace < 0:
        return None, "No '{' found in STDOUT"

    decoder = json.JSONDecoder()
    try:
        obj, _end_idx = decoder.raw_decode(text[first_brace:])
        if isinstance(obj, dict):
            return obj, None
        return None, "First JSON fragment is not an object"
    except json.JSONDecodeError as exc:
        context_start = max(0, exc.pos - 80)
        context_end = min(len(text[first_brace:]), exc.pos + 80)
        snippet = text[first_brace:][context_start:context_end].replace("\n", "\\n")
        return None, f"JSONDecodeError at pos {exc.pos}: {exc.msg}. Context: {snippet}"


def save_inference_log(
    *,
    output_path: Path,
    command: list[str],
    return_code: int,
    stdout: str,
    stderr: str,
    prompt: str,
    model_name: str,
    adapter_path: Path,
    timestamp: str,
    max_tokens: int,
    temperature: float,
    top_p: float,
    generated_json: dict[str, Any] | None,
    parse_error: str | None,
    notes: str = "",
) -> dict[str, Any]:
    """Save inference execution metadata and outputs to JSON."""
    payload: dict[str, Any] = {
        "timestamp_utc": timestamp,
        "model": model_name,
        "adapter_path": str(adapter_path),
        "prompt": prompt,
        "command": command,
        "generation": {
            "max_tokens": max_tokens,
            "temperature": temperature,
            "top_p": top_p,
        },
        "result": {
            "return_code": return_code,
            "stdout": stdout,
            "stderr": stderr,
            "generated_json": generated_json,
            "parse_error": parse_error,
        },
    }
    if notes:
        payload["notes"] = notes

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return payload


if RUN_INFERENCE_TEST:
    model_name_hint = resolve_inference_model_name()
    adapter_path = select_adapter_path(ARTIFACTS_DIR, model_name=model_name_hint)
    if adapter_path is None:
        raise RuntimeError(
            "Adapter LoRA was not found. This notebook does not allow fallback to the base model. "
            "Run the training cell first and verify artifacts under lab/artifacts/<MODEL_ALIAS>/checkpoints_stable."
        )

    artifact_root = adapter_path.parent
    model_name = resolve_model_name_from_artifact_root(artifact_root, fallback=model_name_hint)

    checkpoint_file = select_checkpoint_file(adapter_path, INFERENCE_CHECKPOINT_ITERS)
    if checkpoint_file is not None:
        stage_checkpoint_as_active(adapter_path, checkpoint_file)

    prompt = build_chatml_test_prompt()
    command = build_mlx_generate_command(
        model_name,
        prompt,
        adapter_path=adapter_path,
        max_tokens=INFERENCE_MAX_TOKENS,
        seed=INFERENCE_SEED,
        temp=INFERENCE_TEMP,
        top_p=INFERENCE_TOP_P,
    )
    return_code, stdout_text, stderr_text = run_inference_command(command, ROOT_DIR)
    generated_json, parse_error = extract_generated_json(stdout_text)

    timestamp = datetime.now(UTC).isoformat()
    log_path = artifact_root / "inference_test_log.json"
    payload = save_inference_log(
        output_path=log_path,
        command=command,
        return_code=return_code,
        stdout=stdout_text,
        stderr=stderr_text,
        prompt=prompt,
        model_name=model_name,
        adapter_path=adapter_path,
        timestamp=timestamp,
        max_tokens=INFERENCE_MAX_TOKENS,
        temperature=INFERENCE_TEMP,
        top_p=INFERENCE_TOP_P,
        generated_json=generated_json,
        parse_error=parse_error,
        notes="Adapter-only inference sanity check on complete FHIR Questionnaire generation.",
    )

    print("Inference mode: adapter-only")
    print(f"Repo root: {ROOT_DIR}")
    print(f"Model used: {model_name}")
    print(f"Artifact root: {artifact_root}")
    print(f"Adapter path: {adapter_path}")
    print(f"Inference command: {shlex.join(command)}")
    print(f"Return code: {return_code}")
    print(f"Inference log saved to: {log_path}")
    if stderr_text.strip():
        print("\n--- STDERR (truncated to 1200 chars) ---")
        print(stderr_text[:1200])

    if generated_json is not None:
        print("\n--- GENERATED JSON (pretty) ---")
        print(json.dumps(generated_json, ensure_ascii=False, indent=2))
    else:
        print("\nCould not extract generated JSON from STDOUT.")
        if parse_error:
            print(f"Parse detail: {parse_error}")
        print("\n--- STDOUT PREVIEW (first 1200 chars) ---")
        print(stdout_text[:1200])

    if return_code != 0:
        raise RuntimeError("Inference command failed. Inspect stderr/log for details.")

    payload


Inference mode: adapter-only
Repo root: /Users/CAE9/aiprom-llm
Model used: mlx-community/Qwen2.5-3B-Instruct-4bit
Artifact root: /Users/CAE9/aiprom-llm/lab/artifacts/mlx-community-qwen2.5-3b-instruct-4bit
Adapter path: /Users/CAE9/aiprom-llm/lab/artifacts/mlx-community-qwen2.5-3b-instruct-4bit/checkpoints_stable
Inference command: python -m mlx_lm generate --model mlx-community/Qwen2.5-3B-Instruct-4bit --ignore-chat-template --prompt '<|im_start|>system
You are a FHIR R4 expert. Return ONLY raw JSON (no markdown, no code fences).
Generate VALID FHIR R4 Questionnaire JSON only.
Constraints: do NOT include `initial`; for choice items use `answerOption` as an array of objects, each with one `valueCoding` object containing `system`, `code`, `display`; do NOT nest sub-items under choice; keep it concise.
<|im_end|><|im_start|>user
Generate a complete FHIR Questionnaire for a weekly recovery check-in form. Include resourceType='"'"'Questionnaire'"'"', id='"'"'weekly-recovery-checkin'"'"', st

In [9]:
# Baseline inference test (no adapter): heuristic expected output from base model

from __future__ import annotations

import json
import shlex
import subprocess
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

RUN_BASELINE_INFERENCE = True
BASELINE_SEED = 173
BASELINE_MAX_TOKENS = 900


def resolve_repo_root() -> Path:
    """Best-effort repo root resolver for notebook sessions.

    Avoids accidental `.../lab/lab/artifacts` when the kernel CWD is `lab/`.
    """
    if "repo_root" in globals():
        try:
            candidate = Path(repo_root)
            if candidate.exists():
                return candidate
        except Exception:
            pass

    cwd = Path.cwd().resolve()
    for parent in [cwd, *cwd.parents]:
        if (parent / "requirements.txt").exists() and (parent / "lab").exists():
            return parent

    return cwd


ROOT_DIR = resolve_repo_root()
ARTIFACTS_DIR = Path(artifacts_dir) if "artifacts_dir" in globals() else (ROOT_DIR / "lab" / "artifacts")
DEFAULT_PUBLIC_MODEL = "mlx-community/Qwen2.5-3B-Instruct-4bit"


def load_training_config_from_artifact_root(artifact_root: Path) -> dict[str, Any] | None:
    config_path = artifact_root / "training_config_stable.json"
    if not config_path.exists():
        return None
    try:
        return json.loads(config_path.read_text(encoding="utf-8"))
    except Exception:
        return None


def resolve_artifact_root(artifacts_root: Path) -> Path:
    """Pick the best artifact root directory for logs/config lookup."""
    if (artifacts_root / "training_config_stable.json").exists():
        return artifacts_root

    try:
        for child in artifacts_root.iterdir():
            if child.is_dir() and (child / "training_config_stable.json").exists():
                return child
    except FileNotFoundError:
        pass

    fallback = ROOT_DIR / "lab" / "artifacts"
    if (fallback / "training_config_stable.json").exists():
        return fallback
    try:
        for child in fallback.iterdir():
            if child.is_dir() and (child / "training_config_stable.json").exists():
                return child
    except FileNotFoundError:
        pass

    return artifacts_root


def resolve_baseline_model_name(artifact_root: Path) -> str:
    """Resolve the baseline model name from training config when available."""
    if "training_config" in globals() and isinstance(training_config, dict):
        name = str(training_config.get("model", {}).get("name", "")).strip()
        if name.startswith("mlx-community/"):
            return name

    config = load_training_config_from_artifact_root(artifact_root)
    if isinstance(config, dict):
        model = config.get("model")
        if isinstance(model, dict):
            name = str(model.get("name", "")).strip()
            if name.startswith("mlx-community/"):
                return name

    # Fall back to any already-defined inference model resolver
    if "resolve_inference_model_name" in globals():
        try:
            candidate = str(resolve_inference_model_name()).strip()
            if candidate.startswith("mlx-community/"):
                return candidate
        except Exception:
            pass

    return DEFAULT_PUBLIC_MODEL


def build_baseline_prompt() -> str:
    """A tighter prompt to get a complete, parseable Questionnaire as reference."""
    return (
        "<|im_start|>system\n"
        "You are a FHIR R4 expert. Return ONLY raw JSON (no markdown, no code fences).\n"
        "Generate VALID FHIR R4 Questionnaire JSON only.\n"
        "Constraints: do NOT include `initial`; for choice items use `answerOption` with `valueCoding.display`; do NOT nest sub-items under choice; keep it concise.\n"
        "<|im_end|>"
        "<|im_start|>user\n"
        "Generate a complete FHIR Questionnaire for a weekly recovery check-in form. "
        "Include resourceType='Questionnaire', id='weekly-recovery-checkin', status='active', title, and an item array with exactly 6 questions. "
        "Use mixed item types (choice, string, text, integer, boolean, date) and valid linkId values. "
        "For choice items include 4 answerOption values with valueCoding.display. Return only JSON.\n"
        "<|im_end|>"
        "<|im_start|>assistant\n"
    )


def extract_json_anywhere(text: str) -> dict[str, Any] | None:
    """Best-effort JSON object extraction from model output."""
    s = text.strip()
    fenced = s.find("```json")
    if fenced >= 0:
        s2 = s[fenced + len("```json"):].lstrip()
        fenced_end = s2.rfind("```")
        if fenced_end >= 0:
            s = s2[:fenced_end].strip()
        else:
            s = s2.strip()

    first = s.find("{")
    last = s.rfind("}")
    if 0 <= first < last:
        try:
            obj = json.loads(s[first:last + 1])
            return obj if isinstance(obj, dict) else None
        except Exception:
            return None
    return None


def run_cmd(command: list[str], cwd: Path) -> tuple[int, str, str]:
    """Run a subprocess command and return (return_code, stdout, stderr)."""
    p = subprocess.run(command, cwd=cwd, text=True, capture_output=True, check=False)
    return p.returncode, p.stdout, p.stderr


def save_baseline_log(output_path: Path, payload: dict[str, Any]) -> None:
    """Persist baseline inference payload to disk as formatted JSON."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


if RUN_BASELINE_INFERENCE:
    artifact_root = resolve_artifact_root(ARTIFACTS_DIR)
    model_name = resolve_baseline_model_name(artifact_root)

    prompt = build_baseline_prompt()
    command = [
        "python",
        "-m",
        "mlx_lm",
        "generate",
        "--model",
        model_name,
        "--ignore-chat-template",
        "--prompt",
        prompt,
        "--max-tokens",
        str(BASELINE_MAX_TOKENS),
        "--seed",
        str(BASELINE_SEED),
        "--temp",
        "0.0",
        "--top-p",
        "1.0",
        "--verbose",
        "F",
        "--extra-eos-token",
        "<|im_end|>",
    ]

    return_code, stdout_text, stderr_text = run_cmd(command, ROOT_DIR)
    generated_json = extract_json_anywhere(stdout_text)
    timestamp = datetime.now(UTC).isoformat()
    log_path = artifact_root / "inference_baseline_log.json"

    payload = {
        "timestamp_utc": timestamp,
        "model": model_name,
        "adapter_path": None,
        "prompt": prompt,
        "command": command,
        "generation": {
            "max_tokens": BASELINE_MAX_TOKENS,
            "temperature": 0.0,
            "top_p": 1.0,
            "seed": BASELINE_SEED,
        },
        "result": {
            "return_code": return_code,
            "stdout": stdout_text,
            "stderr": stderr_text,
            "generated_json": generated_json,
        },
        "notes": "Baseline inference reference with base model (no adapter).",
    }
    save_baseline_log(log_path, payload)

    print("Inference mode: baseline (no adapter)")
    print(f"Repo root: {ROOT_DIR}")
    print(f"Artifact root: {artifact_root}")
    print(f"Model used: {model_name}")
    print(f"Inference command: {shlex.join(command)}")
    print(f"Return code: {return_code}")
    print(f"Inference log saved to: {log_path}")
    if stderr_text.strip():
        print("\n--- STDERR (truncated to 1200 chars) ---")
        print(stderr_text[:1200])
    if generated_json is not None:
        print("\n--- GENERATED JSON (pretty) ---")
        print(json.dumps(generated_json, ensure_ascii=False, indent=2))
    else:
        print("\nCould not extract JSON from baseline stdout. Preview stdout (first 600 chars):")
        print(stdout_text[:600])

    if return_code != 0:
        raise RuntimeError("Baseline inference command failed. Inspect stderr/log for details.")

    payload


Inference mode: baseline (no adapter)
Repo root: /Users/CAE9/aiprom-llm
Artifact root: /Users/CAE9/aiprom-llm/lab/artifacts/mlx-community-qwen2.5-3b-instruct-4bit
Model used: mlx-community/Qwen2.5-3B-Instruct-4bit
Inference command: python -m mlx_lm generate --model mlx-community/Qwen2.5-3B-Instruct-4bit --ignore-chat-template --prompt '<|im_start|>system
You are a FHIR R4 expert. Return ONLY raw JSON (no markdown, no code fences).
Generate VALID FHIR R4 Questionnaire JSON only.
Constraints: do NOT include `initial`; for choice items use `answerOption` with `valueCoding.display`; do NOT nest sub-items under choice; keep it concise.
<|im_end|><|im_start|>user
Generate a complete FHIR Questionnaire for a weekly recovery check-in form. Include resourceType='"'"'Questionnaire'"'"', id='"'"'weekly-recovery-checkin'"'"', status='"'"'active'"'"', title, and an item array with exactly 6 questions. Use mixed item types (choice, string, text, integer, boolean, date) and valid linkId values. For 

In [11]:
# A/B check: baseline vs adapter on validation samples (complete Questionnaire)

from __future__ import annotations

import json
import subprocess
from pathlib import Path
from typing import Any

N_SAMPLES = 3
MAX_TOKENS = 768
SEED = 173
TEMP = 0.0
TOP_P = 1.0


def resolve_repo_root() -> Path:
    """Best-effort repo root resolver for notebook sessions."""
    if "repo_root" in globals():
        try:
            candidate = Path(repo_root)
            if candidate.exists():
                return candidate
        except Exception:
            pass

    cwd = Path.cwd().resolve()
    for parent in [cwd, *cwd.parents]:
        if (parent / "requirements.txt").exists() and (parent / "lab").exists():
            return parent

    return cwd


ROOT_DIR = resolve_repo_root()
DATASET_ARTIFACTS_DIR = ROOT_DIR / "lab" / "artifacts"
ARTIFACTS_DIR = Path(artifacts_dir) if "artifacts_dir" in globals() else DATASET_ARTIFACTS_DIR
DEFAULT_PUBLIC_MODEL = "mlx-community/Qwen2.5-3B-Instruct-4bit"


def adapter_artifacts_exist(checkpoint_dir: Path) -> bool:
    markers = {"adapters.safetensors", "adapter_model.safetensors"}
    if not checkpoint_dir.exists() or not checkpoint_dir.is_dir():
        return False

    if any((checkpoint_dir / m).exists() for m in markers):
        return True

    for file in checkpoint_dir.rglob("*"):
        if file.is_file() and file.name in markers:
            return True

    return False


def load_training_config_from_artifact_root(artifact_root: Path) -> dict[str, Any] | None:
    config_path = artifact_root / "training_config_stable.json"
    if not config_path.exists():
        return None
    try:
        return json.loads(config_path.read_text(encoding="utf-8"))
    except Exception:
        return None


def resolve_model_artifact_root() -> Path:
    """Find the model artifact root that contains training_config_stable.json."""
    # Prefer what earlier cells may have set.
    if "artifact_root" in globals():
        try:
            candidate = Path(globals()["artifact_root"])
            if (candidate / "training_config_stable.json").exists():
                return candidate
        except Exception:
            pass

    if (ARTIFACTS_DIR / "training_config_stable.json").exists():
        return ARTIFACTS_DIR

    # Search one level deep under the dataset artifacts directory.
    for base in (ARTIFACTS_DIR, DATASET_ARTIFACTS_DIR):
        try:
            for child in base.iterdir():
                if child.is_dir() and (child / "training_config_stable.json").exists():
                    return child
        except FileNotFoundError:
            pass

    return ARTIFACTS_DIR


def resolve_model_name(model_artifact_root: Path) -> str:
    """Resolve model name for A/B checks, preferring training config."""
    if "training_config" in globals() and isinstance(training_config, dict):
        name = str(training_config.get("model", {}).get("name", "")).strip()
        if name.startswith("mlx-community/"):
            return name

    config = load_training_config_from_artifact_root(model_artifact_root)
    if isinstance(config, dict):
        model = config.get("model")
        if isinstance(model, dict):
            name = str(model.get("name", "")).strip()
            if name.startswith("mlx-community/"):
                return name

    if "resolve_inference_model_name" in globals():
        try:
            candidate = str(resolve_inference_model_name()).strip()
            if candidate.startswith("mlx-community/"):
                return candidate
        except Exception:
            pass

    return DEFAULT_PUBLIC_MODEL


def find_adapter_dir(model_artifact_root: Path) -> Path:
    """Locate adapter directory containing recognized adapter weight files."""
    # Prefer exact checkpoint_dir from training_config.
    if "training_config" in globals() and isinstance(training_config, dict):
        try:
            ck = Path(training_config["outputs"]["checkpoint_dir"])
            if adapter_artifacts_exist(ck):
                return ck
        except Exception:
            pass

    candidates = [
        model_artifact_root / "checkpoints_stable",
        model_artifact_root / "adapters",
        model_artifact_root / "checkpoints",
    ]
    for path in candidates:
        if adapter_artifacts_exist(path):
            return path

    # Also allow nested layouts (just in case)
    for child in candidates:
        if child.exists() and child.is_dir():
            for sub in child.iterdir():
                if sub.is_dir() and adapter_artifacts_exist(sub):
                    return sub

    raise RuntimeError(
        "Adapter directory not found; expected adapters.safetensors under lab/artifacts/<MODEL_ALIAS>/checkpoints_stable"
    )


def parse_chatml_record(text: str) -> tuple[str, str] | None:
    """Return (prompt, expected_json_str) from a ChatML transcript."""
    marker = "<|im_start|>assistant\n"
    if marker not in text:
        return None

    before, after = text.split(marker, 1)
    end = after.find("<|im_end|>")
    expected = after[:end].strip() if end >= 0 else after.strip()

    prompt = before + marker
    return prompt, expected


def run_generate(*, model: str, prompt: str, adapter_dir: Path | None) -> tuple[int, str, str]:
    """Run mlx_lm generation with optional adapter path and deterministic decoding settings."""
    cmd = [
        "python",
        "-m",
        "mlx_lm",
        "generate",
        "--model",
        model,
        "--ignore-chat-template",
        "--prompt",
        prompt,
        "--max-tokens",
        str(MAX_TOKENS),
        "--seed",
        str(SEED),
        "--temp",
        str(TEMP),
        "--top-p",
        str(TOP_P),
        "--verbose",
        "F",
        "--extra-eos-token",
        "<|im_end|>",
    ]
    if adapter_dir is not None:
        cmd.extend(["--adapter-path", str(adapter_dir)])

    p = subprocess.run(cmd, cwd=ROOT_DIR, text=True, capture_output=True)
    return p.returncode, p.stdout, p.stderr


def extract_first_json_object(stdout: str) -> dict[str, Any] | None:
    """Extract the first JSON object from generation stdout."""
    s = stdout.strip()
    start = s.find("{")
    end = s.rfind("}")
    if start < 0 or end <= start:
        return None
    try:
        obj = json.loads(s[start : end + 1])
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None


def count_items(obj: dict[str, Any] | None) -> int:
    """Count all nested Questionnaire items recursively."""
    if not isinstance(obj, dict):
        return 0

    def _walk(items: Any) -> int:
        if not isinstance(items, list):
            return 0
        total = 0
        for node in items:
            if not isinstance(node, dict):
                continue
            total += 1
            total += _walk(node.get("item"))
        return total

    return _walk(obj.get("item"))


def questionnaire_signature(obj: dict[str, Any] | None) -> tuple[str, str, str, int]:
    """Return a compact signature tuple for side-by-side output comparison."""
    if not obj:
        return ("<no-json>", "", "", 0)
    return (
        str(obj.get("resourceType", "")),
        str(obj.get("id", "")),
        str(obj.get("title", ""))[:60],
        count_items(obj),
    )


model_artifact_root = resolve_model_artifact_root()
model = resolve_model_name(model_artifact_root)
adapter_dir = find_adapter_dir(model_artifact_root)

valid_path = DATASET_ARTIFACTS_DIR / "valid.jsonl"
if not valid_path.exists():
    valid_path = DATASET_ARTIFACTS_DIR / "val.jsonl"

lines = valid_path.read_text(encoding="utf-8").splitlines()
records = [json.loads(l) for l in lines[: max(N_SAMPLES, 1)] if l.strip()]

print("A/B: baseline vs adapter (dataset-like prompts)")
print("-" * 60)
print(f"Repo root:   {ROOT_DIR}")
print(f"Data dir:    {DATASET_ARTIFACTS_DIR}")
print(f"Model dir:   {model_artifact_root}")
print(f"Model:       {model}")
print(f"Adapter:     {adapter_dir}")
print(f"Samples:     {min(N_SAMPLES, len(records))}")
print(f"Decoding:    seed={SEED} temp={TEMP} top_p={TOP_P} max_tokens={MAX_TOKENS}")

parseable_baseline = 0
parseable_adapter = 0

resource_match_b = 0
id_match_b = 0
items_match_b = 0

resource_match_a = 0
id_match_a = 0
items_match_a = 0

for i, row in enumerate(records[:N_SAMPLES], start=1):
    text = str(row.get("text", ""))
    parsed = parse_chatml_record(text)
    if parsed is None:
        print(f"\nSample {i}: could not parse ChatML record")
        continue

    prompt, expected_str = parsed
    expected_obj = None
    try:
        expected_obj = json.loads(expected_str)
    except Exception:
        expected_obj = None

    rc_b, out_b, err_b = run_generate(model=model, prompt=prompt, adapter_dir=None)
    rc_a, out_a, err_a = run_generate(model=model, prompt=prompt, adapter_dir=adapter_dir)

    gen_b = extract_first_json_object(out_b) if rc_b == 0 else None
    gen_a = extract_first_json_object(out_a) if rc_a == 0 else None

    if gen_b is not None:
        parseable_baseline += 1
    if gen_a is not None:
        parseable_adapter += 1

    if expected_obj is not None and isinstance(expected_obj, dict):
        exp_items = count_items(expected_obj)

        if gen_b is not None:
            resource_match_b += int(gen_b.get("resourceType") == expected_obj.get("resourceType"))
            id_match_b += int(gen_b.get("id") == expected_obj.get("id"))
            items_match_b += int(count_items(gen_b) == exp_items)

        if gen_a is not None:
            resource_match_a += int(gen_a.get("resourceType") == expected_obj.get("resourceType"))
            id_match_a += int(gen_a.get("id") == expected_obj.get("id"))
            items_match_a += int(count_items(gen_a) == exp_items)

    adapter_loaded_hint = ("adapter" in (err_a or "").lower())

    print(f"\nSample {i}")
    print(f"  adapter_loaded_hint: {adapter_loaded_hint}")
    print(f"  expected: {questionnaire_signature(expected_obj)}")
    print(f"  baseline: {questionnaire_signature(gen_b)}")
    print(f"  adapter:  {questionnaire_signature(gen_a)}")

    if gen_b is None and (err_b or "").strip():
        print("  baseline stderr (head):")
        print("  " + (err_b.strip().splitlines()[0] if err_b.strip().splitlines() else "<empty>"))

    if gen_a is None and (err_a or "").strip():
        print("  adapter stderr (head):")
        print("  " + (err_a.strip().splitlines()[0] if err_a.strip().splitlines() else "<empty>"))

n = min(N_SAMPLES, len(records))
print("\nSummary")
print("-" * 60)
print(f"JSON parseable (baseline): {parseable_baseline}/{n}")
print(f"JSON parseable (adapter):  {parseable_adapter}/{n}")
print(f"resourceType exact match (baseline): {resource_match_b}/{n}")
print(f"id          exact match (baseline): {id_match_b}/{n}")
print(f"item_count  exact match (baseline): {items_match_b}/{n}")
print(f"resourceType exact match (adapter):  {resource_match_a}/{n}")
print(f"id          exact match (adapter):  {id_match_a}/{n}")
print(f"item_count  exact match (adapter):  {items_match_a}/{n}")

print("\nNote: exact matching is strict; prioritize parseability + resourceType + item structure.")


A/B: baseline vs adapter (dataset-like prompts)
------------------------------------------------------------
Repo root:   /Users/CAE9/aiprom-llm
Data dir:    /Users/CAE9/aiprom-llm/lab/artifacts
Model dir:   /Users/CAE9/aiprom-llm/lab/artifacts/mlx-community-qwen2.5-3b-instruct-4bit
Model:       mlx-community/Qwen2.5-3B-Instruct-4bit
Adapter:     /Users/CAE9/aiprom-llm/lab/artifacts/mlx-community-qwen2.5-3b-instruct-4bit/checkpoints_stable
Samples:     3
Decoding:    seed=173 temp=0.0 top_p=1.0 max_tokens=768

Sample 1
  adapter_loaded_hint: False
  expected: ('Questionnaire', 'oncology-en-0007', 'Oncology Symptom Assessment', 5)
  baseline: ('<no-json>', '', '', 0)
  adapter:  ('Questionnaire', 'oncology-en-0306', 'Oncology Symptom Assessment', 5)
  baseline stderr (head):
  Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Sample 2
  adapter_loaded_hint: False
  expected: ('Questionnaire', 'mental_health-en-0009', 'Mental Health Screening', 5)
  baseline: ('<no-json>', '', '',

## Rule-based A/B Evaluation

This section evaluates baseline vs adapter generations against explicit structural constraints derived from the target prompt.

### Evaluation dimensions

- JSON parseability and `resourceType='Questionnaire'`,
- required top-level fields (`id`, `status`, `title`),
- structural constraints on `item` composition and expected type coverage,
- choice-item constraints (`answerOption`, no nested sub-items),
- absence of `initial` fields.

The resulting report is saved as `lab/artifacts/ab_rule_eval.json`.

In [ ]:
# Rule-based batch evaluation: baseline vs adapter against exact prompt constraints

from __future__ import annotations

import json
import subprocess
import sys
from collections import Counter
from pathlib import Path
from typing import Any

EVAL_N_SAMPLES = 30
EVAL_MAX_TOKENS = 900
EVAL_SEED = 173
EVAL_TEMP = 0.0
EVAL_TOP_P = 1.0

ASSISTANT_MARKER = "<|im_start|>assistant\n"
END_MARKER = "<|im_end|>"

REQUIRED_TYPES = {"choice", "string", "text", "integer", "boolean", "date"}


def resolve_repo_root() -> Path:
    """Best-effort repo root resolution for notebook CWD variance."""
    if "repo_root" in globals():
        try:
            return Path(repo_root)
        except Exception:
            pass

    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "lab").is_dir():
            return candidate
    return start


ROOT_DIR = resolve_repo_root()
DATA_ARTIFACTS_DIR = Path(artifacts_dir) if "artifacts_dir" in globals() else (ROOT_DIR / "lab" / "artifacts")


def resolve_model_artifact_root(data_artifacts_dir: Path) -> Path:
    """Locate the model-scoped artifacts folder that contains training_config_stable.json."""
    if "model_artifact_root" in globals():
        try:
            p = Path(model_artifact_root)
            if p.exists():
                return p
        except Exception:
            pass

    if "resolve_model_artifact_root" in globals():
        try:
            p = Path(resolve_model_artifact_root())
            if p.exists():
                return p
        except Exception:
            pass

    candidates = sorted(data_artifacts_dir.glob("*/training_config_stable.json"))
    if not candidates:
        raise FileNotFoundError(
            f"No training_config_stable.json found under {data_artifacts_dir}. "
            "Expected lab/artifacts/<MODEL_ALIAS>/training_config_stable.json"
        )

    newest_cfg = max(candidates, key=lambda p: p.stat().st_mtime)
    return newest_cfg.parent


MODEL_ARTIFACT_ROOT = resolve_model_artifact_root(DATA_ARTIFACTS_DIR)
TRAINING_CONFIG_PATH = MODEL_ARTIFACT_ROOT / "training_config_stable.json"


def read_training_config(path: Path) -> dict[str, Any]:
    data = json.loads(path.read_text(encoding="utf-8"))
    return data if isinstance(data, dict) else {}


TRAINING_CONFIG = read_training_config(TRAINING_CONFIG_PATH)


def resolve_eval_model_name() -> str:
    """Resolve model name for rule-based batch evaluation runs."""
    model_section = TRAINING_CONFIG.get("model")
    if isinstance(model_section, dict):
        name = model_section.get("name")
        if isinstance(name, str) and name.strip():
            return name.strip()

    if "resolve_inference_model_name" in globals():
        try:
            return str(resolve_inference_model_name())
        except Exception:
            pass

    return "mlx-community/Qwen2.5-3B-Instruct-4bit"


def adapter_artifacts_exist(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    markers = ("adapters.safetensors", "adapter_model.safetensors")
    return any(p.name in markers for p in path.rglob("*"))


def resolve_eval_adapter_dir(model_artifact_root: Path) -> Path:
    """Resolve adapter directory from training config or known checkpoint layout."""
    outputs = TRAINING_CONFIG.get("outputs")
    if isinstance(outputs, dict):
        checkpoint_dir = outputs.get("checkpoint_dir")
        if isinstance(checkpoint_dir, str) and checkpoint_dir.strip():
            ckpt = Path(checkpoint_dir)
            candidate = ckpt if ckpt.is_absolute() else (model_artifact_root / ckpt)
            if adapter_artifacts_exist(candidate):
                return candidate

    for candidate in [
        model_artifact_root / "checkpoints_stable",
        model_artifact_root / "checkpoints",
        model_artifact_root / "adapters",
    ]:
        if adapter_artifacts_exist(candidate):
            return candidate

    raise RuntimeError(
        "Adapter directory not found. Expected something like "
        f"{model_artifact_root}/checkpoints_stable containing adapters.safetensors"
    )


def parse_chatml_prompt_and_expected(text: str) -> tuple[str, dict[str, Any] | None] | None:
    """Parse ChatML text into generation prompt plus expected assistant JSON object."""
    if ASSISTANT_MARKER not in text:
        return None

    before, after = text.split(ASSISTANT_MARKER, 1)
    end_idx = after.find(END_MARKER)
    expected_json_str = after[:end_idx].strip() if end_idx >= 0 else after.strip()
    prompt = before + ASSISTANT_MARKER

    try:
        expected_obj = json.loads(expected_json_str)
        if not isinstance(expected_obj, dict):
            expected_obj = None
    except Exception:
        expected_obj = None

    return prompt, expected_obj


def run_generate(*, model: str, prompt: str, adapter_dir: Path | None) -> tuple[int, str, str]:
    """Run mlx_lm generation for evaluation with optional adapter path."""
    cmd = [
        sys.executable,
        "-m",
        "mlx_lm",
        "generate",
        "--model",
        model,
        "--ignore-chat-template",
        "--prompt",
        prompt,
        "--max-tokens",
        str(EVAL_MAX_TOKENS),
        "--seed",
        str(EVAL_SEED),
        "--temp",
        str(EVAL_TEMP),
        "--top-p",
        str(EVAL_TOP_P),
        "--verbose",
        "F",
        "--extra-eos-token",
        "<|im_end|>",
    ]
    if adapter_dir is not None:
        cmd.extend(["--adapter-path", str(adapter_dir)])

    p = subprocess.run(cmd, cwd=ROOT_DIR, text=True, capture_output=True, check=False)
    return p.returncode, p.stdout, p.stderr


def extract_first_json_object(stdout: str) -> dict[str, Any] | None:
    """Extract one JSON object from stdout using robust fallback strategies."""
    text = (stdout or "").strip()
    if not text:
        return None

    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        pass

    fenced_start = text.find("```json")
    if fenced_start >= 0:
        start = fenced_start + len("```json")
        fenced_end = text.find("```", start)
        if fenced_end > start:
            candidate = text[start:fenced_end].strip()
            try:
                parsed = json.loads(candidate)
                if isinstance(parsed, dict):
                    return parsed
            except Exception:
                pass

    first_brace = text.find("{")
    if first_brace < 0:
        return None

    decoder = json.JSONDecoder()
    try:
        obj, _ = decoder.raw_decode(text[first_brace:])
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None


def iter_items(nodes: Any) -> list[dict[str, Any]]:
    """Flatten nested Questionnaire items into a single list."""
    out: list[dict[str, Any]] = []

    def _walk(items: Any) -> None:
        if not isinstance(items, list):
            return
        for node in items:
            if not isinstance(node, dict):
                continue
            out.append(node)
            _walk(node.get("item"))

    _walk(nodes)
    return out


def has_any_initial(node: Any) -> bool:
    """Return True when an `initial` field appears anywhere in a nested structure."""
    if isinstance(node, dict):
        if "initial" in node:
            return True
        return any(has_any_initial(v) for v in node.values())
    if isinstance(node, list):
        return any(has_any_initial(v) for v in node)
    return False


def has_nested_under_choice(item: dict[str, Any]) -> bool:
    """Check whether a choice/open-choice item incorrectly has nested sub-items."""
    if item.get("type") not in {"choice", "open-choice"}:
        return False
    children = item.get("item")
    return isinstance(children, list) and len(children) > 0


def choice_item_has_4_valid_answer_options(item: dict[str, Any]) -> bool:
    """Validate that each choice/open-choice item has exactly four valid answer options."""
    if item.get("type") not in {"choice", "open-choice"}:
        return True

    options = item.get("answerOption")
    if not isinstance(options, list) or len(options) != 4:
        return False

    for option in options:
        if not isinstance(option, dict):
            return False
        value_coding = option.get("valueCoding")
        if not isinstance(value_coding, dict):
            return False
        display = value_coding.get("display")
        if not isinstance(display, str) or not display.strip():
            return False

    return True


def evaluate_rules(obj: dict[str, Any] | None) -> dict[str, bool]:
    """Evaluate strict structural rules for a generated Questionnaire object."""
    rules: dict[str, bool] = {
        "parseable_json": isinstance(obj, dict),
        "resourceType_questionnaire": False,
        "id_present": False,
        "status_active": False,
        "title_present": False,
        "root_item_exactly_6": False,
        "contains_required_types": False,
        "no_initial_anywhere": False,
        "choice_no_nested_subitems": False,
        "choice_has_4_answer_options_with_display": False,
    }

    if not isinstance(obj, dict):
        return rules

    rules["resourceType_questionnaire"] = obj.get("resourceType") == "Questionnaire"
    rules["id_present"] = isinstance(obj.get("id"), str) and bool(obj.get("id", "").strip())
    rules["status_active"] = obj.get("status") == "active"
    rules["title_present"] = isinstance(obj.get("title"), str) and bool(obj.get("title", "").strip())

    root_items = obj.get("item")
    rules["root_item_exactly_6"] = isinstance(root_items, list) and len(root_items) == 6

    all_items = iter_items(root_items)
    types_found = {str(item.get("type")) for item in all_items if isinstance(item.get("type"), str)}
    rules["contains_required_types"] = REQUIRED_TYPES.issubset(types_found)

    rules["no_initial_anywhere"] = not has_any_initial(obj)
    rules["choice_no_nested_subitems"] = all(not has_nested_under_choice(item) for item in all_items)
    rules["choice_has_4_answer_options_with_display"] = all(choice_item_has_4_valid_answer_options(item) for item in all_items)

    return rules


def aggregate_results(rows: list[dict[str, Any]], mode: str) -> dict[str, Any]:
    """Aggregate sample-level rule outcomes into summary pass-rate metrics."""
    selected = [r for r in rows if r["mode"] == mode]
    if not selected:
        return {"mode": mode, "n": 0, "success_rate": 0.0, "rule_pass_rates": {}, "all_rules_pass": 0}

    rule_names = list(selected[0]["rules"].keys())
    counts = Counter()
    all_rules_pass = 0

    for rec in selected:
        if all(rec["rules"].values()):
            all_rules_pass += 1
        for rule in rule_names:
            counts[rule] += int(bool(rec["rules"][rule]))

    n = len(selected)
    rates = {rule: round(counts[rule] * 100.0 / n, 2) for rule in rule_names}

    return {
        "mode": mode,
        "n": n,
        "all_rules_pass": all_rules_pass,
        "success_rate": round(all_rules_pass * 100.0 / n, 2),
        "rule_pass_rates": rates,
    }


model_name = resolve_eval_model_name()
adapter_dir = resolve_eval_adapter_dir(MODEL_ARTIFACT_ROOT)

valid_path = DATA_ARTIFACTS_DIR / "valid.jsonl"
if not valid_path.exists():
    valid_path = DATA_ARTIFACTS_DIR / "val.jsonl"
if not valid_path.exists():
    raise RuntimeError("Could not find valid.jsonl or val.jsonl in lab/artifacts")

all_lines = [line for line in valid_path.read_text(encoding="utf-8").splitlines() if line.strip()]
records = [json.loads(line) for line in all_lines[: max(EVAL_N_SAMPLES, 1)]]

rows: list[dict[str, Any]] = []

for i, row in enumerate(records, start=1):
    text = str(row.get("text", ""))
    parsed = parse_chatml_prompt_and_expected(text)
    if parsed is None:
        continue

    prompt, expected_obj = parsed

    rc_base, out_base, err_base = run_generate(model=model_name, prompt=prompt, adapter_dir=None)
    obj_base = extract_first_json_object(out_base) if rc_base == 0 else None
    rules_base = evaluate_rules(obj_base)
    rows.append({
        "sample": i,
        "mode": "baseline",
        "return_code": rc_base,
        "rules": rules_base,
        "expected_id": expected_obj.get("id") if isinstance(expected_obj, dict) else None,
        "generated_id": obj_base.get("id") if isinstance(obj_base, dict) else None,
        "stderr_head": (err_base or "").splitlines()[:5],
    })

    rc_adp, out_adp, err_adp = run_generate(model=model_name, prompt=prompt, adapter_dir=adapter_dir)
    obj_adp = extract_first_json_object(out_adp) if rc_adp == 0 else None
    rules_adp = evaluate_rules(obj_adp)
    rows.append({
        "sample": i,
        "mode": "adapter",
        "return_code": rc_adp,
        "rules": rules_adp,
        "expected_id": expected_obj.get("id") if isinstance(expected_obj, dict) else None,
        "generated_id": obj_adp.get("id") if isinstance(obj_adp, dict) else None,
        "stderr_head": (err_adp or "").splitlines()[:5],
    })

baseline_summary = aggregate_results(rows, "baseline")
adapter_summary = aggregate_results(rows, "adapter")

rule_names = sorted(set(baseline_summary.get("rule_pass_rates", {}).keys()) | set(adapter_summary.get("rule_pass_rates", {}).keys()))
delta_by_rule = {
    rule: round(adapter_summary["rule_pass_rates"].get(rule, 0.0) - baseline_summary["rule_pass_rates"].get(rule, 0.0), 2)
    for rule in rule_names
}

result_payload = {
    "config": {
        "n_samples": EVAL_N_SAMPLES,
        "max_tokens": EVAL_MAX_TOKENS,
        "seed": EVAL_SEED,
        "temp": EVAL_TEMP,
        "top_p": EVAL_TOP_P,
        "model": model_name,
        "adapter_dir": str(adapter_dir),
        "valid_path": str(valid_path),
        "repo_root": str(ROOT_DIR),
        "data_artifacts_dir": str(DATA_ARTIFACTS_DIR),
        "model_artifact_root": str(MODEL_ARTIFACT_ROOT),
        "training_config": str(TRAINING_CONFIG_PATH),
        "sys_executable": sys.executable,
    },
    "baseline": baseline_summary,
    "adapter": adapter_summary,
    "delta_adapter_minus_baseline": {
        "all_rules_pass_rate": round(adapter_summary["success_rate"] - baseline_summary["success_rate"], 2),
        "by_rule": delta_by_rule,
    },
    "sample_level": rows,
}

report_path = DATA_ARTIFACTS_DIR / "ab_rule_eval.json"
report_path.write_text(json.dumps(result_payload, ensure_ascii=False, indent=2), encoding="utf-8")

print("Rule-based A/B evaluation (baseline vs adapter)")
print("-" * 72)
print(f"Repo root:    {ROOT_DIR}")
print(f"Data dir:     {DATA_ARTIFACTS_DIR}")
print(f"Model dir:    {MODEL_ARTIFACT_ROOT}")
print(f"Model:        {model_name}")
print(f"Adapter dir:  {adapter_dir}")
print(f"Validation:   {valid_path}")
print(f"Samples used: {baseline_summary['n']}")

print("\nALL RULES PASS RATE")
print(f"- Baseline: {baseline_summary['all_rules_pass']}/{baseline_summary['n']} ({baseline_summary['success_rate']}%)")
print(f"- Adapter:  {adapter_summary['all_rules_pass']}/{adapter_summary['n']} ({adapter_summary['success_rate']}%)")
print(f"- Delta:    {result_payload['delta_adapter_minus_baseline']['all_rules_pass_rate']} pp")

print("\nRule pass rates (Adapter - Baseline delta, pp):")
for rule in sorted(delta_by_rule.keys()):
    base_rate = baseline_summary["rule_pass_rates"].get(rule, 0.0)
    adp_rate = adapter_summary["rule_pass_rates"].get(rule, 0.0)
    delta = delta_by_rule[rule]
    print(f"- {rule}: baseline={base_rate}% | adapter={adp_rate}% | delta={delta}")

print(f"\nDetailed report saved to: {report_path}")
result_payload


## Post-analysis: Strict vs Relaxed Criteria

This section re-reads the saved rule-based report and summarizes outcomes under two scoring regimes.

### What each regime means

- **Strict**: all structural rules must pass simultaneously for a sample to count as success.
- **Relaxed**: only core structure rules must pass (parseability, resource type, identity/title presence, and no nested choice sub-items).

### Why both are useful

- **Strict** answers: "Does the model satisfy the full target specification?"
- **Relaxed** answers: "Is the model at least structurally on track, even if detailed constraints still fail?"

### How to interpret deltas (Adapter - Baseline)

- If **both strict and relaxed improve**, the adapter likely adds robust specialization.
- If **relaxed improves but strict does not**, the adapter improves core structure but still fails on fine-grained constraints.
- If **both stagnate or regress**, there is no reliable specialization signal.

Use this view to separate formatting sensitivity from structural behavior and identify dominant failure drivers.

In [ ]:
# Post-analysis from ab_rule_eval.json (no generation): strict vs relaxed view

from __future__ import annotations

import json
from collections import Counter
from pathlib import Path
from typing import Any

ARTIFACTS_DIR = Path(artifacts_dir) if "artifacts_dir" in globals() else (Path(repo_root) / "lab" / "artifacts")
report_path = ARTIFACTS_DIR / "ab_rule_eval.json"
if not report_path.exists():
    raise FileNotFoundError(f"{report_path} does not exist. Run the rule-based evaluation cell first.")

payload = json.loads(report_path.read_text(encoding="utf-8"))
rows: list[dict[str, Any]] = payload.get("sample_level", [])

if not rows:
    raise RuntimeError("ab_rule_eval.json does not contain sample_level")

STRICT_KEYS = [
    "parseable_json",
    "resourceType_questionnaire",
    "id_present",
    "status_active",
    "title_present",
    "root_item_exactly_6",
    "contains_required_types",
    "no_initial_anywhere",
    "choice_no_nested_subitems",
    "choice_has_4_answer_options_with_display",
]

RELAXED_KEYS = [
    "parseable_json",
    "resourceType_questionnaire",
    "id_present",
    "title_present",
    "choice_no_nested_subitems",
]

def summarize(rows: list[dict[str, Any]], mode: str, keys: list[str]) -> dict[str, Any]:
    """Compute strict/relaxed composite pass rates and failure counters for one mode."""
    selected = [r for r in rows if r.get("mode") == mode]
    n = len(selected)
    if n == 0:
        return {"mode": mode, "n": 0, "all_pass": 0, "rate": 0.0, "fail_counter": {}}

    all_pass = 0
    fail_counter: Counter[str] = Counter()

    for rec in selected:
        rules = rec.get("rules", {})
        ok = True
        for k in keys:
            if not bool(rules.get(k, False)):
                fail_counter[k] += 1
                ok = False
        if ok:
            all_pass += 1

    return {
        "mode": mode,
        "n": n,
        "all_pass": all_pass,
        "rate": round(all_pass * 100.0 / n, 2),
        "fail_counter": dict(fail_counter.most_common()),
    }

strict_baseline = summarize(rows, "baseline", STRICT_KEYS)
strict_adapter = summarize(rows, "adapter", STRICT_KEYS)
relaxed_baseline = summarize(rows, "baseline", RELAXED_KEYS)
relaxed_adapter = summarize(rows, "adapter", RELAXED_KEYS)

print("Post-analysis of saved A/B run")
print("-" * 72)
print(f"Report: {report_path}")
print(f"Samples baseline/adapter: {strict_baseline['n']} / {strict_adapter['n']}")

print("\nStrict composite (same as the rule-based evaluation cell)")
print(f"- Baseline: {strict_baseline['all_pass']}/{strict_baseline['n']} ({strict_baseline['rate']}%)")
print(f"- Adapter:  {strict_adapter['all_pass']}/{strict_adapter['n']} ({strict_adapter['rate']}%)")
print(f"- Delta:    {round(strict_adapter['rate'] - strict_baseline['rate'], 2)} pp")

print("\nRelaxed composite (parseable + resourceType + id + title + no nested choice)")
print(f"- Baseline: {relaxed_baseline['all_pass']}/{relaxed_baseline['n']} ({relaxed_baseline['rate']}%)")
print(f"- Adapter:  {relaxed_adapter['all_pass']}/{relaxed_adapter['n']} ({relaxed_adapter['rate']}%)")
print(f"- Delta:    {round(relaxed_adapter['rate'] - relaxed_baseline['rate'], 2)} pp")

print("\nTop strict failure drivers (baseline):")
for k, v in list(strict_baseline["fail_counter"].items())[:5]:
    print(f"- {k}: {v}")

print("\nTop strict failure drivers (adapter):")
for k, v in list(strict_adapter["fail_counter"].items())[:5]:
    print(f"- {k}: {v}")

analysis_payload = {
    "strict": {"baseline": strict_baseline, "adapter": strict_adapter},
    "relaxed": {"baseline": relaxed_baseline, "adapter": relaxed_adapter},
}
analysis_path = ARTIFACTS_DIR / "ab_rule_eval_analysis.json"
analysis_path.write_text(json.dumps(analysis_payload, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nAnalysis saved to: {analysis_path}")
analysis_payload

## Adapter Validation Protocol (Base vs Adapter)

This section formalizes how to decide whether the LoRA adapter adds meaningful specialization over the base model.

### Inputs

- `lab/artifacts/ab_rule_eval.json` (rule-level A/B metrics),
- `lab/artifacts/ab_rule_eval_analysis.json` (strict vs relaxed composite analysis).

### Decision criteria (suggested)

- Adapter `strict` composite pass rate must be **greater than or equal to** baseline strict rate.
- Adapter `relaxed` composite pass rate must improve by at least **+5 percentage points** over baseline.
- Adapter must not regress on core structural rules (`parseable_json`, `resourceType_questionnaire`, `id_present`, `title_present`).

### Practical interpretation

- **Go**: adapter improves relaxed structure quality and does not regress on core constraints.
- **Conditional Go**: small gains, no critical regressions; keep runtime validator + retry.
- **No-Go**: no measurable gain or structural regressions.

In [ ]:
# Go/No-Go decision from saved A/B artifacts (no new inference)

from __future__ import annotations

import json
from pathlib import Path
from typing import Any

ARTIFACTS_DIR = Path(artifacts_dir) if "artifacts_dir" in globals() else (Path(repo_root) / "lab" / "artifacts")
rule_eval_path = ARTIFACTS_DIR / "ab_rule_eval.json"
analysis_path = ARTIFACTS_DIR / "ab_rule_eval_analysis.json"

if not rule_eval_path.exists():
    raise FileNotFoundError(f"Missing {rule_eval_path}. Run the rule-based evaluation section first.")
if not analysis_path.exists():
    raise FileNotFoundError(f"Missing {analysis_path}. Run the post-analysis section first.")

rule_eval = json.loads(rule_eval_path.read_text(encoding="utf-8"))
analysis = json.loads(analysis_path.read_text(encoding="utf-8"))

baseline_rule_rates: dict[str, float] = rule_eval.get("baseline", {}).get("rule_pass_rates", {})
adapter_rule_rates: dict[str, float] = rule_eval.get("adapter", {}).get("rule_pass_rates", {})

strict_baseline = float(analysis.get("strict", {}).get("baseline", {}).get("rate", 0.0))
strict_adapter = float(analysis.get("strict", {}).get("adapter", {}).get("rate", 0.0))
relaxed_baseline = float(analysis.get("relaxed", {}).get("baseline", {}).get("rate", 0.0))
relaxed_adapter = float(analysis.get("relaxed", {}).get("adapter", {}).get("rate", 0.0))

strict_delta = round(strict_adapter - strict_baseline, 2)
relaxed_delta = round(relaxed_adapter - relaxed_baseline, 2)

CORE_RULES = [
    "parseable_json",
    "resourceType_questionnaire",
    "id_present",
    "title_present",
]

core_regressions: list[str] = []
for rule in CORE_RULES:
    b = float(baseline_rule_rates.get(rule, 0.0))
    a = float(adapter_rule_rates.get(rule, 0.0))
    if a < b:
        core_regressions.append(f"{rule}: adapter={a}% < baseline={b}%")

strict_ok = strict_adapter >= strict_baseline
relaxed_ok = relaxed_delta >= 5.0
core_ok = len(core_regressions) == 0

if strict_ok and relaxed_ok and core_ok:
    decision = "GO"
elif core_ok and (strict_delta >= 0 or relaxed_delta >= 0):
    decision = "CONDITIONAL_GO"
else:
    decision = "NO_GO"

summary: dict[str, Any] = {
    "decision": decision,
    "strict": {
        "baseline_rate": strict_baseline,
        "adapter_rate": strict_adapter,
        "delta_pp": strict_delta,
    },
    "relaxed": {
        "baseline_rate": relaxed_baseline,
        "adapter_rate": relaxed_adapter,
        "delta_pp": relaxed_delta,
    },
    "core_rule_regressions": core_regressions,
    "criteria": {
        "strict_non_regression": strict_ok,
        "relaxed_delta_ge_5pp": relaxed_ok,
        "no_core_rule_regression": core_ok,
    },
    "artifacts": {
        "rule_eval_path": str(rule_eval_path),
        "analysis_path": str(analysis_path),
    },
}

summary_path = ARTIFACTS_DIR / "adapter_go_no_go.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("Adapter acceptance decision")
print("-" * 72)
print(f"Decision: {decision}")
print(f"Strict delta:  {strict_delta} pp")
print(f"Relaxed delta: {relaxed_delta} pp")
print(f"Core regressions: {len(core_regressions)}")
if core_regressions:
    for reg in core_regressions:
        print(f"- {reg}")
print(f"Saved to: {summary_path}")

summary

## Quantitative Evaluation

This section computes deterministic post-training quality indicators, including:

- dataset-level JSON validity,
- FHIR structural pass-rate,
- type-wise support and conformance,
- summary tables for reporting.

When no model generations are available yet, this section still reports gold-dataset structural metrics as a baseline.

In [ ]:
# Compute quantitative summary tables

from __future__ import annotations

from collections import Counter
from typing import Any


def iter_questionnaire_items(items: Any) -> list[dict[str, Any]]:
    """Flatten questionnaire items recursively."""
    flat: list[dict[str, Any]] = []

    def _walk(nodes: Any) -> None:
        """Recursively traverse nested item arrays and collect item nodes."""
        if not isinstance(nodes, list):
            return
        for node in nodes:
            if not isinstance(node, dict):
                continue
            flat.append(node)
            children = node.get("item")
            if isinstance(children, list) and children:
                _walk(children)

    _walk(items)
    return flat


def build_type_quality_table(records: list[RecordParseResult]) -> list[dict[str, Any]]:
    """Build per-item-type structural quality summary using item-level checks."""
    totals: Counter[str] = Counter()
    passed: Counter[str] = Counter()

    for rec in records:
        for item in iter_questionnaire_items(rec.assistant_obj.get("item")):
            item_type = item.get("type")
            key = item_type if isinstance(item_type, str) else "<missing-or-invalid>"
            totals[key] += 1
            if not audit_questionnaire_item(item):
                passed[key] += 1

    rows: list[dict[str, Any]] = []
    for t in sorted(totals.keys()):
        n_total = totals[t]
        n_pass = passed[t]
        n_fail = n_total - n_pass
        pass_rate = (n_pass / n_total * 100.0) if n_total else 0.0
        rows.append(
            {
                "type": t,
                "total": n_total,
                "pass": n_pass,
                "fail": n_fail,
                "pass_rate_percent": round(pass_rate, 2),
            }
        )
    return rows


def pretty_print_table(rows: list[dict[str, Any]], title: str) -> None:
    """Print a compact plain-text table."""
    print(title)
    print("-" * 60)
    if not rows:
        print("<empty>")
        return

    headers = list(rows[0].keys())
    widths = {h: max(len(h), max(len(str(r[h])) for r in rows)) for h in headers}

    header_line = " | ".join(h.ljust(widths[h]) for h in headers)
    sep_line = "-+-".join("-" * widths[h] for h in headers)
    print(header_line)
    print(sep_line)

    for r in rows:
        print(" | ".join(str(r[h]).ljust(widths[h]) for h in headers))


quality_rows = build_type_quality_table(parsed_records)
pretty_print_table(quality_rows, "FHIR structural quality by item type")

overall_pass = sum(r["pass"] for r in quality_rows)
overall_total = sum(r["total"] for r in quality_rows)
overall_rate = (overall_pass / overall_total * 100.0) if overall_total else 0.0

print("\nOverall structural pass rate")
print("-" * 60)
print(f"Pass: {overall_pass}/{overall_total} ({overall_rate:.2f}%)")

## Qualitative Analysis

This section inspects representative records to support narrative error analysis and reporting.

### Goals

- highlight examples across different FHIR item types,
- inspect structural variety,
- prepare templates for future model-output comparison.

In a later training run, this section can be extended to compare gold references versus generated predictions side by side.

In [ ]:
# Sample qualitative examples by FHIR item type (from complete Questionnaires)

from __future__ import annotations

import json
import random
from collections import defaultdict
from typing import Any


def iter_questionnaire_items_with_context(payload: dict[str, Any]) -> list[tuple[str, dict[str, Any]]]:
    """Return (questionnaire_id, item) tuples for all nested items."""
    questionnaire_id = str(payload.get("id", "<missing-id>"))
    flat: list[tuple[str, dict[str, Any]]] = []

    def _walk(nodes: Any) -> None:
        """Recursively traverse items while preserving questionnaire context."""
        if not isinstance(nodes, list):
            return
        for node in nodes:
            if not isinstance(node, dict):
                continue
            flat.append((questionnaire_id, node))
            children = node.get("item")
            if isinstance(children, list) and children:
                _walk(children)

    _walk(payload.get("item"))
    return flat


def sample_examples_by_type(
    records: list[RecordParseResult],
    samples_per_type: int,
    seed: int,
) -> dict[str, list[dict[str, Any]]]:
    """Collect deterministic random item samples grouped by item type."""
    buckets: dict[str, list[dict[str, Any]]] = defaultdict(list)

    for rec in records:
        for questionnaire_id, item in iter_questionnaire_items_with_context(rec.assistant_obj):
            item_type = item.get("type")
            key = item_type if isinstance(item_type, str) else "<missing-or-invalid>"
            buckets[key].append(
                {
                    "questionnaire_id": questionnaire_id,
                    "item": item,
                }
            )

    rng = random.Random(seed)
    sampled: dict[str, list[dict[str, Any]]] = {}
    for key in sorted(buckets.keys()):
        choices = buckets[key][:]
        rng.shuffle(choices)
        sampled[key] = choices[:samples_per_type]

    return sampled


def print_qualitative_samples(samples: dict[str, list[dict[str, Any]]]) -> None:
    """Pretty-print sampled qualitative records grouped by item type."""
    for item_type, rows in samples.items():
        print(f"\nType: {item_type}")
        print("-" * 60)
        if not rows:
            print("<no samples>")
            continue
        for idx, row in enumerate(rows, start=1):
            item = row["item"]
            preview = {
                "questionnaire_id": row["questionnaire_id"],
                "linkId": item.get("linkId"),
                "text": item.get("text"),
                "type": item.get("type"),
                "required": item.get("required"),
            }
            print(f"Sample {idx}: {json.dumps(preview, ensure_ascii=False)}")


qual_samples = sample_examples_by_type(
    records=parsed_records,
    samples_per_type=2,
    seed=GLOBAL_SEED,
)
print_qualitative_samples(qual_samples)

## Limitations, Ethics, and Threats to Validity

### Methodological Limitations

- Even with 1,500 synthetic records, overfitting can still occur if prompt patterns are narrow.
- Structural validity does not guarantee semantic clinical correctness.
- Prompt distribution may not represent all real-world questionnaire authoring styles.

### External Validity

- Results obtained on this dataset may not transfer to other institutions, languages, or proprietary form libraries without adaptation.

### Ethical and Legal Considerations

- Some PROM/PREM instruments have third-party licensing constraints.
- This notebook is intended for research/educational usage; production or redistribution may require additional permissions.
- Generated clinical forms should be reviewed by qualified professionals before operational use.

### Reproducibility Caveat

- Exact deterministic behavior can vary across platform versions and backend implementations, despite fixed seeds and documented configs.

## References

### Standards and Interoperability

- HL7 FHIR R4 Questionnaire: https://hl7.org/fhir/R4/questionnaire.html
- FAIR Principles: https://www.go-fair.org/fair-principles/

### Model Adaptation

- LoRA (Hu et al., 2021): https://arxiv.org/abs/2106.09685
- QLoRA (Dettmers et al., 2023): https://arxiv.org/abs/2305.14314

### Tooling

- MLX: https://github.com/ml-explore/mlx
- MLX-LM: https://github.com/ml-explore/mlx-lm
- Qwen docs: https://qwen.readthedocs.io/
- CodeCarbon documentation: https://mlco2.github.io/codecarbon/

### Clinical Instrument Context

- PROMIS (HealthMeasures): https://www.healthmeasures.net/explore-measurement-systems/promis
- EORTC terms and conditions: https://qol.eortc.org/terms-conditions/academic-user/
- EQ-5D access and registration: https://euroqol.org/register/obtain-eq-5d/how-to-obtain-eq-5d/

---

End of notebook.

## How to Cite

If you use this notebook or derived artifacts in academic work, please cite the repository and the methodological references below.

### Suggested citation (repository)

```text
Pimàs, P. (2026). aiprom-llm: Model-agnostic fine-tuning for FHIR R4 Questionnaire generation [Computer software]. GitHub.
```

### Suggested citation (methods)

- Hu, E. J., et al. (2021). LoRA: Low-Rank Adaptation of Large Language Models. https://arxiv.org/abs/2106.09685
- Dettmers, T., et al. (2023). QLoRA: Efficient Finetuning of Quantized LLMs. https://arxiv.org/abs/2305.14314
- HL7 FHIR R4 Questionnaire specification. https://hl7.org/fhir/R4/questionnaire.html

### Reproducibility citation note

For reproducibility claims, include:

- commit hash,
- training configuration file,
- split manifest,
- dataset manifest,
- runtime platform metadata.

These artifacts are generated in `lab/artifacts/` by this notebook workflow.